**Public archival copy.** Saved cell outputs (including embedded images) and Colab-specific runtime metadata have been removed. Original code and explanatory cell content remain unchanged. Some code refers to Google Drive paths and intermediate research artifacts that are not distributed here. The Fitzpatrick17k image files and the researchers' individual access link are **not included**. See `notebooks/README.md` for access and reproducibility notes.


# Corrected Phase 5: Mechanistic Analysis of Skin-Tone Domain Shift and Learned Representations

## Thesis

**Domain-Invariant and Explainable Deep Learning for Equitable and Generalizable Skin Lesion Malignancy Classification Across Diverse Skin Tones**

---

## Purpose

This notebook rebuilds Phase 5 using the corrected and frozen Phase 4 models.

The corrected Phase 4 experiment contains three controlled training compositions:

| Experiment | Fair | Dark | Benign | Malignant | Total |
|---|---:|---:|---:|---:|---:|
| M1_75F_25D | 600 | 200 | 400 | 400 | 800 |
| M2_50F_50D | 400 | 400 | 400 | 400 | 800 |
| M3_25F_75D | 200 | 600 | 400 | 400 | 800 |

Five independently trained models were retained per composition:

$$
n=5
$$

with seeds:

$$
42,\ 43,\ 44,\ 45,\ 46
$$

giving:

$$
3\times5=15
$$

corrected Phase 4 models.

Phase 5 performs **no model retraining**.

It investigates:

1. prediction probabilities,
2. confidence,
3. predictive entropy,
4. logits,
5. calibration and probabilistic error,
6. malignant false-negative persistence,
7. learned avgpool representations,
8. class-conditional Fair–Dark representation discrepancy,
9. PCA visualization.

The independent experimental unit remains the seed-specific trained model, not the individual image.

The original Phase 5 artifacts remain preserved as a historical implementation branch.

In [ ]:
# ============================================
# Section 1.1
# Import Core Libraries
# ============================================

import os
import json
import math
import random
import itertools
import warnings

from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    DataLoader
)

from torchvision import (
    transforms,
    models
)

from PIL import Image

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    brier_score_loss
)

warnings.filterwarnings("ignore")

print(
    "PyTorch version:",
    torch.__version__
)

print(
    "CUDA available:",
    torch.cuda.is_available()
)

In [ ]:
# ============================================
# Section 1.2
# Mount Google Drive
# ============================================

from google.colab import drive

drive.mount(
    "/content/drive"
)

print(
    "✓ Google Drive mounted."
)

In [ ]:
# ============================================
# Section 1.3
# Define Compute Device
# ============================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print(
    "Current device:",
    DEVICE
)


if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

else:

    print(
        "CPU mode active."
    )

    print(
        "Phase 5 setup and statistics can proceed on CPU."
    )

# Section 2: Corrected Phase 5 Artifact Configuration

The original and corrected mechanistic analyses are stored separately.

The original Phase 5 remains unchanged:

```text
Phase_5_Mechanistic_Analysis/

In [ ]:

## Code Cell 6
# ============================================
# Section 2.1
# Define Core Artifact Paths
# ============================================

ARTIFACT_PATH = Path(
    "/content/drive/MyDrive/Phase_3_Artifacts"
)

DATA_PATH = (
    ARTIFACT_PATH
    / "dataset_splits"
)


# --------------------------------------------
# Original Phase 5
# --------------------------------------------

ORIGINAL_PHASE5_PATH = (
    ARTIFACT_PATH
    / "Phase_5_Mechanistic_Analysis"
)


# --------------------------------------------
# Corrected Phase 4
# --------------------------------------------

CORRECTED_PHASE4_PATH = (
    ARTIFACT_PATH
    / "Phase_4_Mixing_Experiments_Corrected"
)

CORRECTED_PHASE4_MODEL_PATH = (
    CORRECTED_PHASE4_PATH
    / "models"
)

CORRECTED_PHASE4_TABLE_PATH = (
    CORRECTED_PHASE4_PATH
    / "tables"
)

CORRECTED_PHASE4_MANIFEST_PATH = (
    CORRECTED_PHASE4_PATH
    / "manifests"
)


CORRECTED_PHASE4_RESULTS_FILE = (
    CORRECTED_PHASE4_TABLE_PATH
    / "Phase4_Corrected_Repeated_Run_Results.csv"
)

CORRECTED_PHASE4_COMPLETION_FILE = (
    CORRECTED_PHASE4_MANIFEST_PATH
    / "Phase4_Corrected_Completion_Manifest.json"
)


print(
    "Corrected Phase 4 root:"
)

print(
    CORRECTED_PHASE4_PATH
)

In [ ]:
# ============================================
# Section 2.2
# Define Corrected Phase 5 Artifact Paths
# ============================================

CORRECTED_PHASE5_PATH = (
    ARTIFACT_PATH
    / "Phase_5_Mechanistic_Analysis_Corrected"
)

PHASE5_PREDICTION_PATH = (
    CORRECTED_PHASE5_PATH
    / "prediction_level"
)

PHASE5_TABLE_PATH = (
    CORRECTED_PHASE5_PATH
    / "tables"
)

PHASE5_FIGURE_PATH = (
    CORRECTED_PHASE5_PATH
    / "figures"
)

PHASE5_REPRESENTATION_PATH = (
    CORRECTED_PHASE5_PATH
    / "representations"
)

PHASE5_PCA_PATH = (
    CORRECTED_PHASE5_PATH
    / "representation_pca"
)

PHASE5_MANIFEST_PATH = (
    CORRECTED_PHASE5_PATH
    / "manifests"
)

PHASE5_AUDIT_PATH = (
    CORRECTED_PHASE5_PATH
    / "audits"
)


phase5_directories = [

    CORRECTED_PHASE5_PATH,

    PHASE5_PREDICTION_PATH,

    PHASE5_TABLE_PATH,

    PHASE5_FIGURE_PATH,

    PHASE5_REPRESENTATION_PATH,

    PHASE5_PCA_PATH,

    PHASE5_MANIFEST_PATH,

    PHASE5_AUDIT_PATH
]


for path in phase5_directories:

    path.mkdir(
        parents=True,
        exist_ok=True
    )


print(
    "✓ Corrected Phase 5 directory structure created."
)

print(
    "\nRoot:"
)

print(
    CORRECTED_PHASE5_PATH
)

In [ ]:
# ============================================
# Section 2.3
# Verify Required Upstream Artifacts
# ============================================

required_paths = {

    "Dataset splits":
        DATA_PATH,

    "Corrected Phase 4 root":
        CORRECTED_PHASE4_PATH,

    "Corrected checkpoints":
        CORRECTED_PHASE4_MODEL_PATH,

    "Corrected Phase 4 results":
        CORRECTED_PHASE4_RESULTS_FILE,

    "Corrected Phase 4 manifest":
        CORRECTED_PHASE4_COMPLETION_FILE,

    "Corrected Phase 5 root":
        CORRECTED_PHASE5_PATH
}


print("=" * 95)

print(
    "CORRECTED PHASE 5 UPSTREAM ARTIFACT AUDIT"
)

print("=" * 95)


all_required_found = True


for name, path in required_paths.items():

    exists = path.exists()

    if not exists:

        all_required_found = False


    status = (
        "✓ FOUND"
        if exists
        else "✗ MISSING"
    )


    print(
        f"{name:<35} {status}"
    )


print("=" * 95)


if all_required_found:

    print(
        "✓ All required upstream artifacts are available."
    )

else:

    print(
        "✗ One or more required upstream artifacts are missing."
    )

In [ ]:
# ============================================
# Section 2.4
# Load and Verify Corrected Phase 4 Completion Manifest
# ============================================

with open(
    CORRECTED_PHASE4_COMPLETION_FILE,
    "r"
) as f:

    phase4_manifest = json.load(f)


print("=" * 90)

print(
    "CORRECTED PHASE 4 COMPLETION MANIFEST"
)

print("=" * 90)


print(
    "Phase:",
    phase4_manifest.get(
        "phase"
    )
)

print(
    "Status:",
    phase4_manifest.get(
        "status"
    )
)

print(
    "Models trained:",
    phase4_manifest.get(
        "models_trained"
    )
)

print(
    "Checkpoint audit passed:",
    phase4_manifest.get(
        "checkpoint_audit_passed"
    )
)

print(
    "Metric audit passed:",
    phase4_manifest.get(
        "metric_audit_passed"
    )
)

print(
    "Train/evaluation overlap:",
    phase4_manifest.get(
        "train_validation_test_overlap"
    )
)

print(
    "Holm-supported outcomes:",
    phase4_manifest.get(
        "holm_supported_outcomes"
    )
)

In [ ]:
# ============================================
# Section 2.5
# Load Corrected Phase 4 Master Results
# ============================================

phase4_results_df = pd.read_csv(
    CORRECTED_PHASE4_RESULTS_FILE
)


print(
    "Corrected Phase 4 results shape:",
    phase4_results_df.shape
)


print(
    "\nRuns per experiment:"
)

print(
    phase4_results_df[
        "Experiment"
    ]
    .value_counts()
    .sort_index()
)


print(
    "\nSeeds:"
)

for experiment_name in [

    "M1_75F_25D",
    "M2_50F_50D",
    "M3_25F_75D"

]:

    seeds = sorted(

        phase4_results_df.loc[
            phase4_results_df[
                "Experiment"
            ]
            ==
            experiment_name,
            "Seed"
        ]
        .astype(int)
        .tolist()
    )


    print(
        experiment_name,
        seeds
    )

In [ ]:
# ============================================
# Section 2.6
# Audit All 15 Corrected Phase 4 Checkpoints
# ============================================

EXPERIMENTS = [

    "M1_75F_25D",

    "M2_50F_50D",

    "M3_25F_75D"
]


SEEDS = [
    42,
    43,
    44,
    45,
    46
]


checkpoint_rows = []


for experiment_name in EXPERIMENTS:

    for seed in SEEDS:

        checkpoint_file = (

            CORRECTED_PHASE4_MODEL_PATH

            / f"{experiment_name}_seed_{seed}.pth"
        )


        checkpoint_rows.append({

            "Experiment":
                experiment_name,

            "Seed":
                seed,

            "Checkpoint":
                checkpoint_file.name,

            "Exists":
                checkpoint_file.exists(),

            "Size_MB":
                (
                    checkpoint_file.stat().st_size
                    / (1024 ** 2)
                )
                if checkpoint_file.exists()
                else np.nan
        })


phase5_checkpoint_audit_df = (
    pd.DataFrame(
        checkpoint_rows
    )
)


display(
    phase5_checkpoint_audit_df
)


print(
    "\nCheckpoints found:",
    int(
        phase5_checkpoint_audit_df[
            "Exists"
        ].sum()
    ),
    "/ 15"
)


if (
    len(
        phase5_checkpoint_audit_df
    ) == 15

    and

    phase5_checkpoint_audit_df[
        "Exists"
    ]
    .all()
):

    print(
        "✓ All 15 corrected Phase 4 checkpoints are available."
    )

else:

    print(
        "✗ Checkpoint audit failed."
    )

## Section 2.7: Corrected Phase 5 Analysis Plan

The corrected mechanistic analysis will follow the same predefined analytical structure as the original Phase 5:

### Stage A — Reproduction Audit
Reconstruct corrected Phase 4 predictions directly from:

$$
\text{checkpoint}
\rightarrow
\text{architecture}
\rightarrow
\text{preprocessing}
\rightarrow
\text{test image}
\rightarrow
\text{prediction}
\rightarrow
\text{metric}
$$

The expected number of prediction records is:

$$
15\times(487+162)
=
9735
$$

### Stage B — Prediction Behavior
Analyze:

- malignant probability,
- logits,
- confidence,
- predictive entropy,
- class-conditional Fair–Dark differences.

### Stage C — Calibration
Analyze:

- Brier score,
- class-conditional probabilistic error,
- Fair–Dark calibration differences.

### Stage D — Error Persistence
Analyze malignant false negatives across:

- compositions,
- seeds,
- individual images.

### Stage E — Learned Representations
Extract final ResNet50 avgpool representations:

$$
z\in\mathbb{R}^{2048}
$$

for Fair and Dark test images.

### Stage F — Class-Conditional MMD
Compare Fair and Dark feature distributions separately for:

$$
Y=0
$$

and:

$$
Y=1
$$

using a common class-specific Gaussian-kernel bandwidth.

### Stage G — PCA
Fit PCA separately within each trained model.

PCA is used only as a descriptive visualization and not as inferential evidence.

### Stage H — Statistical Synthesis
The independent unit is the trained seed-specific model.

Exact repeated-measures permutation tests and multiplicity correction will be retained where applicable.

# Section 3: Frozen Test Data and Model Reconstruction

Corrected Phase 5 must reproduce the corrected Phase 4 inference pipeline exactly.

The frozen Fair and Dark test sets remain unchanged:

## Fair test set

$$
N_F = 487
$$

with:

$$
N_{F,benign}=239
$$

and:

$$
N_{F,malignant}=248
$$

## Dark test set

$$
N_D = 162
$$

with:

$$
N_{D,benign}=86
$$

and:

$$
N_{D,malignant}=76
$$

For each of the 15 corrected models, predictions will be reconstructed directly from:

$$
\text{checkpoint}
\rightarrow
\text{ResNet50}
\rightarrow
\text{frozen preprocessing}
\rightarrow
\text{fixed test data}
$$

In [ ]:
# ============================================
# Section 3.1
# Load Frozen Fair and Dark Test Splits
# ============================================

FAIR_TEST_FILE = (
    DATA_PATH
    / "fair_test.csv"
)

DARK_TEST_FILE = (
    DATA_PATH
    / "dark_test.csv"
)


fair_test_df = pd.read_csv(
    FAIR_TEST_FILE
)

dark_test_df = pd.read_csv(
    DARK_TEST_FILE
)


print("=" * 90)
print("FROZEN TEST SETS")
print("=" * 90)

print(
    "Fair test shape:",
    fair_test_df.shape
)

print(
    "Dark test shape:",
    dark_test_df.shape
)


print(
    "\nFair target distribution:"
)

print(
    fair_test_df[
        "target"
    ]
    .value_counts()
    .sort_index()
)


print(
    "\nDark target distribution:"
)

print(
    dark_test_df[
        "target"
    ]
    .value_counts()
    .sort_index()
)

In [ ]:
# ============================================
# Section 3.2
# Verify Frozen Test-Set Integrity
# ============================================

test_audit_rows = []


for split_name, df in {

    "Fair Test":
        fair_test_df,

    "Dark Test":
        dark_test_df

}.items():

    test_audit_rows.append({

        "Split":
            split_name,

        "Total":
            len(df),

        "Benign":
            int(
                df["target"]
                .eq(0)
                .sum()
            ),

        "Malignant":
            int(
                df["target"]
                .eq(1)
                .sum()
            ),

        "Duplicate_MD5":
            int(
                df["md5hash"]
                .duplicated()
                .sum()
            ),

        "Duplicate_Path":
            int(
                df["image_path"]
                .duplicated()
                .sum()
            ),

        "Missing_Critical":
            int(
                df[
                    [
                        "md5hash",
                        "image_path",
                        "skin_tone",
                        "target"
                    ]
                ]
                .isna()
                .sum()
                .sum()
            )
    })


phase5_test_audit_df = (
    pd.DataFrame(
        test_audit_rows
    )
)


display(
    phase5_test_audit_df
)


md5_overlap = len(

    set(
        fair_test_df[
            "md5hash"
        ]
    )

    &

    set(
        dark_test_df[
            "md5hash"
        ]
    )
)


print(
    "Fair-Dark MD5 overlap:",
    md5_overlap
)

In [ ]:
# ============================================
# Section 3.3
# Load Frozen Preprocessing Configuration
# ============================================

PREPROCESSING_CONFIG_FILE = (
    ARTIFACT_PATH
    / "preprocessing_config.json"
)


with open(
    PREPROCESSING_CONFIG_FILE,
    "r"
) as f:

    preprocessing_config = json.load(
        f
    )


IMAGE_SIZE = (
    preprocessing_config[
        "image_size"
    ]
)

NORMALIZE_MEAN = (
    preprocessing_config[
        "normalization"
    ]["mean"]
)

NORMALIZE_STD = (
    preprocessing_config[
        "normalization"
    ]["std"]
)


print(
    "Image size:",
    IMAGE_SIZE
)

print(
    "Mean:",
    NORMALIZE_MEAN
)

print(
    "Std:",
    NORMALIZE_STD
)

In [ ]:
# ============================================
# Section 3.4
# Define Frozen Evaluation Transform
# ============================================

eval_transform = transforms.Compose([

    transforms.Resize(
        (
            IMAGE_SIZE,
            IMAGE_SIZE
        )
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=NORMALIZE_MEAN,
        std=NORMALIZE_STD
    )
])


print(
    "✓ Frozen evaluation transform created."
)

In [ ]:
# ============================================
# Section 3.5
# Define Test Dataset
# ============================================

class ClassificationDataset(
    Dataset
):

    def __init__(
        self,
        dataframe,
        transform=None
    ):

        self.dataframe = (
            dataframe
            .reset_index(
                drop=True
            )
        )

        self.transform = transform


    def __len__(
        self
    ):

        return len(
            self.dataframe
        )


    def __getitem__(
        self,
        idx
    ):

        row = (
            self.dataframe
            .iloc[idx]
        )


        image = Image.open(
            row[
                "image_path"
            ]
        ).convert(
            "RGB"
        )


        if self.transform:

            image = (
                self.transform(
                    image
                )
            )


        label = torch.tensor(
            int(
                row[
                    "target"
                ]
            ),
            dtype=torch.long
        )


        return (
            image,
            label,
            idx
        )

In [ ]:
# ============================================
# Section 3.6
# Create Frozen Test DataLoaders
# ============================================

BATCH_SIZE = 32


fair_test_dataset = (
    ClassificationDataset(
        fair_test_df,
        transform=eval_transform
    )
)


dark_test_dataset = (
    ClassificationDataset(
        dark_test_df,
        transform=eval_transform
    )
)


fair_test_loader = DataLoader(

    fair_test_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=torch.cuda.is_available()
)


dark_test_loader = DataLoader(

    dark_test_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=torch.cuda.is_available()
)


print(
    "Fair test dataset:",
    len(
        fair_test_dataset
    )
)

print(
    "Dark test dataset:",
    len(
        dark_test_dataset
    )
)

In [ ]:
# ============================================
# Section 3.7
# Define Corrected Phase 4 ResNet50 Architecture
# ============================================

NUM_CLASSES = 2


def create_phase4_resnet50():

    model = models.resnet50(
        weights=None
    )


    num_features = (
        model.fc
        .in_features
    )


    model.fc = nn.Linear(
        num_features,
        NUM_CLASSES
    )


    model = model.to(
        DEVICE
    )


    return model

In [ ]:
# ============================================
# Section 3.8
# Load One Corrected Checkpoint
# ============================================

TEST_EXPERIMENT = (
    "M1_75F_25D"
)

TEST_SEED = 42


TEST_CHECKPOINT = (

    CORRECTED_PHASE4_MODEL_PATH

    / f"{TEST_EXPERIMENT}_seed_{TEST_SEED}.pth"
)


test_model = (
    create_phase4_resnet50()
)


state_dict = torch.load(
    TEST_CHECKPOINT,
    map_location=DEVICE
)


test_model.load_state_dict(
    state_dict
)


test_model.eval()


print(
    "✓ Checkpoint loaded:"
)

print(
    TEST_CHECKPOINT.name
)


print(
    "\nFinal classifier:"
)

print(
    test_model.fc
)

In [ ]:
# ============================================
# Section 3.9
# Run One Forward-Pass Sanity Check
# ============================================

images, labels, indices = next(
    iter(
        fair_test_loader
    )
)


images = images.to(
    DEVICE
)


with torch.no_grad():

    logits = test_model(
        images
    )

    probabilities = torch.softmax(
        logits,
        dim=1
    )


print(
    "Image batch:",
    images.shape
)

print(
    "Logits:",
    logits.shape
)

print(
    "Probabilities:",
    probabilities.shape
)

print(
    "Probability row sums:"
)

print(
    probabilities[
        :5
    ]
    .sum(
        dim=1
    )
    .cpu()
    .numpy()
)


print(
    "Labels:",
    labels[
        :10
    ]
    .numpy()
)

print(
    "Indices:",
    indices[
        :10
    ]
    .numpy()
)

In [ ]:
# ============================================
# Section 3.10
# Release Sanity-Check Model
# ============================================

del test_model

if torch.cuda.is_available():

    torch.cuda.empty_cache()


print(
    "✓ Sanity-check model released."
)

# Section 4: Prediction-Level Reconstruction

Predictions are reconstructed independently from all 15 corrected Phase 4 checkpoints.

For every model-image pair, the following variables are retained:

- experiment,
- seed,
- test domain,
- source image index,
- image path,
- MD5 hash,
- true class,
- benign logit,
- malignant logit,
- benign probability,
- malignant probability,
- predicted class,
- correctness,
- prediction confidence,
- signed logit difference,
- absolute logit difference,
- predictive entropy.

The signed logit difference is:

$$
\Delta o=o_1-o_0
$$

and binary predictive entropy is:

$$
H=-\sum_{k=0}^{1}p_k\log(p_k)
$$

The expected total number of prediction records is:

$$
15\times(487+162)=9735
$$

In [ ]:
# ============================================
# Section 4.1
# Define Prediction Extraction Function
# ============================================

def extract_predictions(
    model,
    dataloader,
    source_df,
    domain_name,
    experiment_name,
    seed
):

    model.eval()

    records = []


    with torch.no_grad():

        for images, labels, indices in dataloader:

            images = images.to(
                DEVICE
            )

            labels_device = labels.to(
                DEVICE
            )


            logits = model(
                images
            )


            probabilities = torch.softmax(
                logits,
                dim=1
            )


            predictions = torch.argmax(
                probabilities,
                dim=1
            )


            confidence = torch.max(
                probabilities,
                dim=1
            ).values


            signed_logit_difference = (
                logits[:, 1]
                -
                logits[:, 0]
            )


            absolute_logit_difference = (
                signed_logit_difference.abs()
            )


            entropy = -torch.sum(

                probabilities
                *
                torch.log(
                    probabilities.clamp_min(
                        1e-12
                    )
                ),

                dim=1
            )


            logits_cpu = (
                logits
                .cpu()
                .numpy()
            )

            probabilities_cpu = (
                probabilities
                .cpu()
                .numpy()
            )

            predictions_cpu = (
                predictions
                .cpu()
                .numpy()
            )

            confidence_cpu = (
                confidence
                .cpu()
                .numpy()
            )

            signed_difference_cpu = (
                signed_logit_difference
                .cpu()
                .numpy()
            )

            absolute_difference_cpu = (
                absolute_logit_difference
                .cpu()
                .numpy()
            )

            entropy_cpu = (
                entropy
                .cpu()
                .numpy()
            )

            labels_cpu = (
                labels
                .cpu()
                .numpy()
            )

            indices_cpu = (
                indices
                .cpu()
                .numpy()
            )


            for batch_position, source_index in enumerate(
                indices_cpu
            ):

                row = source_df.iloc[
                    int(source_index)
                ]


                record = {

                    "Experiment":
                        experiment_name,

                    "Seed":
                        int(seed),

                    "Domain":
                        domain_name,

                    "Source_Index":
                        int(source_index),

                    "image_path":
                        row["image_path"],

                    "md5hash":
                        row["md5hash"],

                    "skin_tone":
                        row["skin_tone"],

                    "True_Label":
                        int(
                            labels_cpu[
                                batch_position
                            ]
                        ),

                    "Benign_Logit":
                        float(
                            logits_cpu[
                                batch_position,
                                0
                            ]
                        ),

                    "Malignant_Logit":
                        float(
                            logits_cpu[
                                batch_position,
                                1
                            ]
                        ),

                    "Benign_Probability":
                        float(
                            probabilities_cpu[
                                batch_position,
                                0
                            ]
                        ),

                    "Malignant_Probability":
                        float(
                            probabilities_cpu[
                                batch_position,
                                1
                            ]
                        ),

                    "Predicted_Label":
                        int(
                            predictions_cpu[
                                batch_position
                            ]
                        ),

                    "Correct":
                        int(
                            predictions_cpu[
                                batch_position
                            ]
                            ==
                            labels_cpu[
                                batch_position
                            ]
                        ),

                    "Confidence":
                        float(
                            confidence_cpu[
                                batch_position
                            ]
                        ),

                    "Signed_Logit_Difference":
                        float(
                            signed_difference_cpu[
                                batch_position
                            ]
                        ),

                    "Absolute_Logit_Difference":
                        float(
                            absolute_difference_cpu[
                                batch_position
                            ]
                        ),

                    "Predictive_Entropy":
                        float(
                            entropy_cpu[
                                batch_position
                            ]
                        )
                }


                records.append(
                    record
                )


    return pd.DataFrame(
        records
    )

In [ ]:
# ============================================
# Section 4.2
# Define Resume-Safe Prediction Artifact Paths
# ============================================

def prediction_file_path(
    experiment_name,
    seed
):

    return (

        PHASE5_PREDICTION_PATH

        / (
            f"{experiment_name}"
            f"_seed_{seed}"
            f"_predictions.csv"
        )
    )


def prediction_file_complete(
    experiment_name,
    seed
):

    file_path = prediction_file_path(
        experiment_name,
        seed
    )


    if not file_path.exists():

        return False


    try:

        df = pd.read_csv(
            file_path
        )

    except Exception:

        return False


    expected_rows = (
        len(fair_test_df)
        +
        len(dark_test_df)
    )


    required_columns = {

        "Experiment",
        "Seed",
        "Domain",
        "Source_Index",
        "image_path",
        "md5hash",
        "True_Label",

        "Benign_Logit",
        "Malignant_Logit",

        "Benign_Probability",
        "Malignant_Probability",

        "Predicted_Label",
        "Correct",
        "Confidence",

        "Signed_Logit_Difference",
        "Absolute_Logit_Difference",
        "Predictive_Entropy"
    }


    return (

        len(df)
        ==
        expected_rows

        and

        required_columns.issubset(
            df.columns
        )

        and

        df["Experiment"]
        .eq(
            experiment_name
        )
        .all()

        and

        df["Seed"]
        .eq(
            seed
        )
        .all()
    )

In [ ]:
# ============================================
# Section 4.3
# Test Prediction Extraction on One Model
# ============================================

TEST_EXPERIMENT = "M1_75F_25D"
TEST_SEED = 42


test_checkpoint = (

    CORRECTED_PHASE4_MODEL_PATH

    / f"{TEST_EXPERIMENT}_seed_{TEST_SEED}.pth"
)


test_model = (
    create_phase4_resnet50()
)


test_state_dict = torch.load(

    test_checkpoint,

    map_location=DEVICE
)


test_model.load_state_dict(
    test_state_dict
)


test_model.eval()


test_fair_predictions = (
    extract_predictions(

        model=test_model,

        dataloader=fair_test_loader,

        source_df=fair_test_df,

        domain_name="Fair",

        experiment_name=TEST_EXPERIMENT,

        seed=TEST_SEED
    )
)


print(
    "Fair prediction rows:",
    len(
        test_fair_predictions
    )
)


display(
    test_fair_predictions.head()
)


print(
    "\nProbability sum range:"
)

probability_sums = (

    test_fair_predictions[
        "Benign_Probability"
    ]

    +

    test_fair_predictions[
        "Malignant_Probability"
    ]
)


print(
    probability_sums.min(),
    probability_sums.max()
)


del test_model

if torch.cuda.is_available():

    torch.cuda.empty_cache()

In [ ]:
# ============================================
# Section 4.4
# Define Prediction-Level Consistency Audit
# ============================================

def audit_prediction_dataframe(
    prediction_df
):

    probability_sum = (

        prediction_df[
            "Benign_Probability"
        ]

        +

        prediction_df[
            "Malignant_Probability"
        ]
    )


    reconstructed_probability = (

        1.0

        /

        (
            1.0
            +
            np.exp(
                -prediction_df[
                    "Signed_Logit_Difference"
                ]
            )
        )
    )


    predicted_from_probability = (

        prediction_df[
            "Malignant_Probability"
        ]
        >=
        0.5
    ).astype(int)


    return {

        "Rows":
            len(
                prediction_df
            ),

        "Max_Probability_Sum_Error":
            float(
                np.max(
                    np.abs(
                        probability_sum
                        -
                        1.0
                    )
                )
            ),

        "Max_Logit_Probability_Error":
            float(
                np.max(
                    np.abs(
                        reconstructed_probability
                        -
                        prediction_df[
                            "Malignant_Probability"
                        ]
                    )
                )
            ),

        "Prediction_Mismatch_Count":
            int(
                np.sum(
                    predicted_from_probability
                    !=
                    prediction_df[
                        "Predicted_Label"
                    ]
                )
            ),

        "Entropy_Min":
            float(
                prediction_df[
                    "Predictive_Entropy"
                ]
                .min()
            ),

        "Entropy_Max":
            float(
                prediction_df[
                    "Predictive_Entropy"
                ]
                .max()
            )
    }

In [ ]:
# ============================================
# Section 4.5
# Audit Test Prediction Extraction
# ============================================

test_prediction_audit = (
    audit_prediction_dataframe(
        test_fair_predictions
    )
)


for key, value in (
    test_prediction_audit.items()
):

    print(
        f"{key}: {value}"
    )

In [ ]:
# ============================================
# Section 4.6
# Extract Predictions from All 15 Corrected Models
# Resume-Safe
# ============================================

prediction_run_summary = []


for experiment_name in EXPERIMENTS:

    for seed in SEEDS:

        output_file = (
            prediction_file_path(
                experiment_name,
                seed
            )
        )


        if prediction_file_complete(
            experiment_name,
            seed
        ):

            print(
                f"✓ SKIP: "
                f"{experiment_name} "
                f"seed {seed}"
            )

            prediction_run_summary.append({

                "Experiment":
                    experiment_name,

                "Seed":
                    seed,

                "Status":
                    "Existing",

                "Prediction_File":
                    str(
                        output_file
                    )
            })

            continue


        print(
            "\n" + "=" * 90
        )

        print(
            f"Processing "
            f"{experiment_name} | "
            f"Seed {seed}"
        )

        print(
            "=" * 90
        )


        checkpoint_file = (

            CORRECTED_PHASE4_MODEL_PATH

            / (
                f"{experiment_name}"
                f"_seed_{seed}.pth"
            )
        )


        model = (
            create_phase4_resnet50()
        )


        state_dict = torch.load(

            checkpoint_file,

            map_location=DEVICE
        )


        model.load_state_dict(
            state_dict
        )


        model.eval()


        fair_predictions = (
            extract_predictions(

                model=model,

                dataloader=fair_test_loader,

                source_df=fair_test_df,

                domain_name="Fair",

                experiment_name=experiment_name,

                seed=seed
            )
        )


        dark_predictions = (
            extract_predictions(

                model=model,

                dataloader=dark_test_loader,

                source_df=dark_test_df,

                domain_name="Dark",

                experiment_name=experiment_name,

                seed=seed
            )
        )


        model_predictions = (
            pd.concat(

                [
                    fair_predictions,
                    dark_predictions
                ],

                ignore_index=True
            )
        )


        model_audit = (
            audit_prediction_dataframe(
                model_predictions
            )
        )


        assert (
            len(
                model_predictions
            )
            ==
            649
        )


        assert (
            model_audit[
                "Prediction_Mismatch_Count"
            ]
            ==
            0
        )


        model_predictions.to_csv(

            output_file,

            index=False
        )


        prediction_run_summary.append({

            "Experiment":
                experiment_name,

            "Seed":
                seed,

            "Status":
                "Generated",

            "Prediction_File":
                str(
                    output_file
                )
        })


        print(
            "✓ Saved 649 predictions."
        )


        del (
            model,
            state_dict,
            fair_predictions,
            dark_predictions,
            model_predictions
        )


        if torch.cuda.is_available():

            torch.cuda.empty_cache()


prediction_run_summary_df = (
    pd.DataFrame(
        prediction_run_summary
    )
)


display(
    prediction_run_summary_df
)

In [ ]:
# ============================================
# Section 4.7
# Combine All Prediction-Level Files
# ============================================

all_prediction_dfs = []


for experiment_name in EXPERIMENTS:

    for seed in SEEDS:

        file_path = (
            prediction_file_path(
                experiment_name,
                seed
            )
        )


        if not prediction_file_complete(
            experiment_name,
            seed
        ):

            raise RuntimeError(

                f"Incomplete prediction file: "
                f"{experiment_name}, seed {seed}"
            )


        run_df = pd.read_csv(
            file_path
        )


        all_prediction_dfs.append(
            run_df
        )


all_predictions_df = pd.concat(

    all_prediction_dfs,

    ignore_index=True
)


ALL_PREDICTIONS_FILE = (

    PHASE5_PREDICTION_PATH

    / "Phase5_Corrected_All_Model_Predictions.csv"
)


all_predictions_df.to_csv(

    ALL_PREDICTIONS_FILE,

    index=False
)


print(
    "Combined prediction shape:",
    all_predictions_df.shape
)


print(
    "Expected rows:",
    15 * 649
)


print(
    "Saved:"
)

print(
    ALL_PREDICTIONS_FILE
)

In [ ]:
# ============================================
# Section 4.8
# Audit Complete Prediction Dataset
# ============================================

print("=" * 95)

print(
    "COMPLETE CORRECTED PHASE 5 PREDICTION AUDIT"
)

print("=" * 95)


print(
    "Total records:",
    len(
        all_predictions_df
    )
)


print(
    "\nRecords by experiment:"
)

print(

    all_predictions_df[
        "Experiment"
    ]
    .value_counts()
    .sort_index()
)


print(
    "\nRecords by domain:"
)

print(

    all_predictions_df[
        "Domain"
    ]
    .value_counts()
)


print(
    "\nModels represented:",
    all_predictions_df[
        [
            "Experiment",
            "Seed"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)


full_prediction_audit = (
    audit_prediction_dataframe(
        all_predictions_df
    )
)


print(
    "\nMathematical consistency:"
)


for key, value in (
    full_prediction_audit.items()
):

    print(
        f"{key}: {value}"
    )

In [ ]:
# ============================================
# Section 4.9
# Save Prediction Reconstruction Audit
# ============================================

prediction_audit_record = {

    "Total_Prediction_Records":
        len(
            all_predictions_df
        ),

    "Expected_Prediction_Records":
        9735,

    "Number_of_Models":
        all_predictions_df[
            [
                "Experiment",
                "Seed"
            ]
        ]
        .drop_duplicates()
        .shape[0],

    "Fair_Records":
        int(
            all_predictions_df[
                "Domain"
            ]
            .eq(
                "Fair"
            )
            .sum()
        ),

    "Dark_Records":
        int(
            all_predictions_df[
                "Domain"
            ]
            .eq(
                "Dark"
            )
            .sum()
        ),

    **full_prediction_audit
}


phase5_prediction_audit_df = pd.DataFrame(
    [
        prediction_audit_record
    ]
)


phase5_prediction_audit_df.to_csv(

    PHASE5_AUDIT_PATH
    / "Phase5_Corrected_Prediction_Reconstruction_Audit.csv",

    index=False
)


display(
    phase5_prediction_audit_df
)

# Section 5: End-to-End Corrected Phase 4 Reproduction Audit

Before interpreting prediction uncertainty, calibration, errors, or learned representations, Corrected Phase 5 independently reconstructs the corrected Phase 4 performance metrics from the saved prediction-level outputs.

For every seed-specific model, metrics are recomputed separately on the frozen Fair and Dark test domains.

The following metrics are reconstructed:

- accuracy,
- precision,
- recall,
- F1 score,
- AUC,
- specificity,
- false-positive rate,
- false-negative rate.

The resulting values are then compared against the frozen corrected Phase 4 master-results table.

This provides an end-to-end audit of:

$$
\text{checkpoint}
\rightarrow
\text{prediction}
\rightarrow
\text{metric}
$$

In [ ]:
# ============================================
# Section 5.1
# Define Metric Reconstruction Function
# ============================================

def compute_binary_metrics(
    prediction_df
):

    y_true = (
        prediction_df[
            "True_Label"
        ]
        .astype(int)
        .values
    )

    y_pred = (
        prediction_df[
            "Predicted_Label"
        ]
        .astype(int)
        .values
    )

    y_prob = (
        prediction_df[
            "Malignant_Probability"
        ]
        .astype(float)
        .values
    )


    accuracy = accuracy_score(
        y_true,
        y_pred
    )


    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )


    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )


    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )


    auc = roc_auc_score(
        y_true,
        y_prob
    )


    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()


    specificity = (
        tn
        /
        (tn + fp)
    )


    fpr = (
        fp
        /
        (fp + tn)
    )


    fnr = (
        fn
        /
        (fn + tp)
    )


    return {

        "Accuracy":
            float(
                accuracy
            ),

        "Precision":
            float(
                precision
            ),

        "Recall":
            float(
                recall
            ),

        "F1":
            float(
                f1
            ),

        "AUC":
            float(
                auc
            ),

        "Specificity":
            float(
                specificity
            ),

        "FPR":
            float(
                fpr
            ),

        "FNR":
            float(
                fnr
            ),

        "TN":
            int(tn),

        "FP":
            int(fp),

        "FN":
            int(fn),

        "TP":
            int(tp)
    }

In [ ]:
# ============================================
# Section 5.2
# Recompute Fair and Dark Metrics
# For All 15 Corrected Models
# ============================================

recomputed_rows = []


for experiment_name in EXPERIMENTS:

    for seed in SEEDS:

        run_df = (
            all_predictions_df[

                (
                    all_predictions_df[
                        "Experiment"
                    ]
                    ==
                    experiment_name
                )

                &

                (
                    all_predictions_df[
                        "Seed"
                    ]
                    ==
                    seed
                )
            ]
        )


        fair_df = (
            run_df[
                run_df[
                    "Domain"
                ]
                ==
                "Fair"
            ]
        )


        dark_df = (
            run_df[
                run_df[
                    "Domain"
                ]
                ==
                "Dark"
            ]
        )


        fair_metrics = (
            compute_binary_metrics(
                fair_df
            )
        )


        dark_metrics = (
            compute_binary_metrics(
                dark_df
            )
        )


        row = {

            "Experiment":
                experiment_name,

            "Seed":
                seed
        }


        for metric_name, value in (
            fair_metrics.items()
        ):

            row[
                f"Fair_{metric_name}"
            ] = value


        for metric_name, value in (
            dark_metrics.items()
        ):

            row[
                f"Dark_{metric_name}"
            ] = value


        row["Accuracy_Gap"] = abs(
            fair_metrics["Accuracy"]
            -
            dark_metrics["Accuracy"]
        )


        row["Recall_Gap"] = abs(
            fair_metrics["Recall"]
            -
            dark_metrics["Recall"]
        )


        row["F1_Gap"] = abs(
            fair_metrics["F1"]
            -
            dark_metrics["F1"]
        )


        row["AUC_Gap"] = abs(
            fair_metrics["AUC"]
            -
            dark_metrics["AUC"]
        )


        row["FNR_Gap"] = abs(
            fair_metrics["FNR"]
            -
            dark_metrics["FNR"]
        )


        recomputed_rows.append(
            row
        )


phase5_recomputed_metrics_df = (
    pd.DataFrame(
        recomputed_rows
    )
)


display(
    phase5_recomputed_metrics_df.head()
)


print(
    "\nShape:",
    phase5_recomputed_metrics_df.shape
)

In [ ]:
# ============================================
# Section 5.3
# Save Recomputed Corrected Phase 4 Metrics
# ============================================

RECOMPUTED_METRICS_FILE = (

    PHASE5_TABLE_PATH

    / "Phase5_Corrected_Recomputed_Phase4_Metrics.csv"
)


phase5_recomputed_metrics_df.to_csv(

    RECOMPUTED_METRICS_FILE,

    index=False
)


print(
    "✓ Saved:"
)

print(
    RECOMPUTED_METRICS_FILE
)

In [ ]:
# ============================================
# Section 5.4
# Compare Recomputed Metrics Against
# Frozen Corrected Phase 4 Master Results
# ============================================

metrics_to_compare = [

    "Fair_Accuracy",
    "Fair_Precision",
    "Fair_Recall",
    "Fair_F1",
    "Fair_AUC",
    "Fair_Specificity",
    "Fair_FPR",
    "Fair_FNR",

    "Dark_Accuracy",
    "Dark_Precision",
    "Dark_Recall",
    "Dark_F1",
    "Dark_AUC",
    "Dark_Specificity",
    "Dark_FPR",
    "Dark_FNR",

    "Accuracy_Gap",
    "Recall_Gap",
    "F1_Gap",
    "AUC_Gap",
    "FNR_Gap"
]


comparison_rows = []


for _, phase4_row in (
    phase4_results_df.iterrows()
):

    experiment_name = (
        phase4_row[
            "Experiment"
        ]
    )

    seed = int(
        phase4_row[
            "Seed"
        ]
    )


    reconstructed_row = (
        phase5_recomputed_metrics_df[

            (
                phase5_recomputed_metrics_df[
                    "Experiment"
                ]
                ==
                experiment_name
            )

            &

            (
                phase5_recomputed_metrics_df[
                    "Seed"
                ]
                ==
                seed
            )
        ]
        .iloc[0]
    )


    for metric in metrics_to_compare:

        phase4_value = float(
            phase4_row[
                metric
            ]
        )


        reconstructed_value = float(
            reconstructed_row[
                metric
            ]
        )


        comparison_rows.append({

            "Experiment":
                experiment_name,

            "Seed":
                seed,

            "Metric":
                metric,

            "Phase4_Value":
                phase4_value,

            "Phase5_Reconstructed":
                reconstructed_value,

            "Absolute_Difference":
                abs(
                    phase4_value
                    -
                    reconstructed_value
                )
        })


phase5_phase4_reproduction_audit_df = (
    pd.DataFrame(
        comparison_rows
    )
)


display(
    phase5_phase4_reproduction_audit_df.head(
        20
    )
)

In [ ]:
# ============================================
# Section 5.5
# Summarize Phase 4 Reproduction Accuracy
# ============================================

max_difference = (

    phase5_phase4_reproduction_audit_df[
        "Absolute_Difference"
    ]
    .max()
)


mean_difference = (

    phase5_phase4_reproduction_audit_df[
        "Absolute_Difference"
    ]
    .mean()
)


comparisons_above_1e10 = int(

    (
        phase5_phase4_reproduction_audit_df[
            "Absolute_Difference"
        ]
        >
        1e-10
    )
    .sum()
)


comparisons_above_1e8 = int(

    (
        phase5_phase4_reproduction_audit_df[
            "Absolute_Difference"
        ]
        >
        1e-8
    )
    .sum()
)


print("=" * 90)

print(
    "CORRECTED PHASE 4 REPRODUCTION AUDIT"
)

print("=" * 90)


print(
    "Metric comparisons:",
    len(
        phase5_phase4_reproduction_audit_df
    )
)


print(
    "Maximum absolute difference:",
    max_difference
)


print(
    "Mean absolute difference:",
    mean_difference
)


print(
    "Differences > 1e-10:",
    comparisons_above_1e10
)


print(
    "Differences > 1e-8:",
    comparisons_above_1e8
)

In [ ]:
# ============================================
# Section 5.6
# Audit Reconstructed Fair-Dark Performance Gaps
# ============================================

gap_metrics = [

    "Accuracy_Gap",
    "Recall_Gap",
    "F1_Gap",
    "AUC_Gap",
    "FNR_Gap"
]


gap_reproduction_audit_df = (

    phase5_phase4_reproduction_audit_df[

        phase5_phase4_reproduction_audit_df[
            "Metric"
        ]
        .isin(
            gap_metrics
        )
    ]

    .copy()
)


gap_summary_df = (

    gap_reproduction_audit_df

    .groupby(
        "Metric"
    )[
        "Absolute_Difference"
    ]

    .agg(
        [
            "max",
            "mean"
        ]
    )

    .reset_index()
)


display(
    gap_summary_df
)

In [ ]:
# ============================================
# Section 5.7
# Save Corrected Phase 4 Reproduction Audits
# ============================================

phase5_phase4_reproduction_audit_df.to_csv(

    PHASE5_AUDIT_PATH
    / "Phase5_Corrected_Phase4_Reproduction_Audit.csv",

    index=False
)


gap_reproduction_audit_df.to_csv(

    PHASE5_AUDIT_PATH
    / "Phase5_Corrected_Phase4_Gap_Reproduction_Audit.csv",

    index=False
)


gap_summary_df.to_csv(

    PHASE5_AUDIT_PATH
    / "Phase5_Corrected_Phase4_Gap_Reproduction_Summary.csv",

    index=False
)


print(
    "✓ Corrected Phase 4 reproduction audits saved."
)

## Section 5.8: Reproduction-Audit Interpretation Rule

Corrected Phase 5 mechanistic analyses are considered valid only if the independently reconstructed metrics reproduce the frozen corrected Phase 4 results to numerical precision.

Minor discrepancies on the order of floating-point rounding are acceptable.

A successful audit demonstrates that:

- the correct checkpoints were loaded,
- the correct test images were used,
- preprocessing was reproduced correctly,
- class probabilities were reconstructed correctly,
- prediction labels were reconstructed correctly,
- Phase 5 is analyzing exactly the same model behavior summarized in corrected Phase 4.

Only after this audit passes will the mechanistic analyses be interpreted.

In [ ]:
# ============================================
# Section 5.9
# Locate Non-Numerically-Exact Reproduction Results
# ============================================

non_exact_reproduction_df = (

    phase5_phase4_reproduction_audit_df[

        phase5_phase4_reproduction_audit_df[
            "Absolute_Difference"
        ]
        >
        1e-10
    ]

    .copy()
)


non_exact_reproduction_df = (

    non_exact_reproduction_df

    .sort_values(
        "Absolute_Difference",
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


display(
    non_exact_reproduction_df
)


print(
    "Number of differences > 1e-10:",
    len(
        non_exact_reproduction_df
    )
)


print(
    "\nAffected metrics:"
)

print(
    non_exact_reproduction_df[
        "Metric"
    ]
    .value_counts()
)

In [ ]:
# ============================================
# Section 5.10
# Audit AUC Numerical Precision
# Probability vs Signed Logit Difference
# ============================================

auc_precision_rows = []


for experiment_name in EXPERIMENTS:

    for seed in SEEDS:

        for domain_name in [
            "Fair",
            "Dark"
        ]:

            subset = (

                all_predictions_df[

                    (
                        all_predictions_df[
                            "Experiment"
                        ]
                        ==
                        experiment_name
                    )

                    &

                    (
                        all_predictions_df[
                            "Seed"
                        ]
                        ==
                        seed
                    )

                    &

                    (
                        all_predictions_df[
                            "Domain"
                        ]
                        ==
                        domain_name
                    )
                ]
            )


            y_true = (
                subset[
                    "True_Label"
                ]
                .astype(int)
                .values
            )


            probability_auc = roc_auc_score(

                y_true,

                subset[
                    "Malignant_Probability"
                ]
                .astype(float)
                .values
            )


            logit_auc = roc_auc_score(

                y_true,

                subset[
                    "Signed_Logit_Difference"
                ]
                .astype(float)
                .values
            )


            phase4_column = (
                f"{domain_name}_AUC"
            )


            phase4_auc = float(

                phase4_results_df.loc[

                    (
                        phase4_results_df[
                            "Experiment"
                        ]
                        ==
                        experiment_name
                    )

                    &

                    (
                        phase4_results_df[
                            "Seed"
                        ]
                        ==
                        seed
                    ),

                    phase4_column
                ]
                .iloc[0]
            )


            auc_precision_rows.append({

                "Experiment":
                    experiment_name,

                "Seed":
                    seed,

                "Domain":
                    domain_name,

                "Phase4_AUC":
                    phase4_auc,

                "Probability_AUC":
                    probability_auc,

                "Logit_AUC":
                    logit_auc,

                "Probability_Difference":
                    abs(
                        probability_auc
                        -
                        phase4_auc
                    ),

                "Logit_Difference":
                    abs(
                        logit_auc
                        -
                        phase4_auc
                    )
            })


auc_precision_audit_df = (
    pd.DataFrame(
        auc_precision_rows
    )
)


display(

    auc_precision_audit_df[

        (
            auc_precision_audit_df[
                "Probability_Difference"
            ]
            >
            1e-10
        )

        |

        (
            auc_precision_audit_df[
                "Logit_Difference"
            ]
            >
            1e-10
        )
    ]
)

In [ ]:
# ============================================
# Section 5.11
# Inspect Probability Saturation / Ranking Ties
# ============================================

tie_rows = []


for experiment_name in EXPERIMENTS:

    for seed in SEEDS:

        for domain_name in [
            "Fair",
            "Dark"
        ]:

            subset = (

                all_predictions_df[

                    (
                        all_predictions_df[
                            "Experiment"
                        ]
                        ==
                        experiment_name
                    )

                    &

                    (
                        all_predictions_df[
                            "Seed"
                        ]
                        ==
                        seed
                    )

                    &

                    (
                        all_predictions_df[
                            "Domain"
                        ]
                        ==
                        domain_name
                    )
                ]

                .copy()
            )


            probability_duplicates = int(

                subset[
                    "Malignant_Probability"
                ]
                .duplicated(
                    keep=False
                )
                .sum()
            )


            logit_duplicates = int(

                subset[
                    "Signed_Logit_Difference"
                ]
                .duplicated(
                    keep=False
                )
                .sum()
            )


            tie_rows.append({

                "Experiment":
                    experiment_name,

                "Seed":
                    seed,

                "Domain":
                    domain_name,

                "Probability_Tied_Rows":
                    probability_duplicates,

                "Logit_Tied_Rows":
                    logit_duplicates
            })


auc_tie_audit_df = (
    pd.DataFrame(
        tie_rows
    )
)


display(

    auc_tie_audit_df[

        (
            auc_tie_audit_df[
                "Probability_Tied_Rows"
            ]
            >
            0
        )

        |

        (
            auc_tie_audit_df[
                "Logit_Tied_Rows"
            ]
            >
            0
        )
    ]
)

In [ ]:
# ============================================
# Section 5.12
# Verify Discrete AUC Ranking Resolution
# ============================================

FAIR_POSITIVES = int(
    fair_test_df[
        "target"
    ]
    .eq(1)
    .sum()
)

FAIR_NEGATIVES = int(
    fair_test_df[
        "target"
    ]
    .eq(0)
    .sum()
)


fair_auc_pair_increment = (

    1.0
    /
    (
        FAIR_POSITIVES
        *
        FAIR_NEGATIVES
    )
)


fair_auc_half_pair_increment = (

    fair_auc_pair_increment
    /
    2.0
)


print(
    "Fair positives:",
    FAIR_POSITIVES
)

print(
    "Fair negatives:",
    FAIR_NEGATIVES
)

print(
    "One ranking-pair AUC increment:",
    fair_auc_pair_increment
)

print(
    "Half ranking-pair AUC increment:",
    fair_auc_half_pair_increment
)


affected_auc_df = (

    non_exact_reproduction_df[

        non_exact_reproduction_df[
            "Metric"
        ]
        ==
        "Fair_AUC"
    ]

    .copy()
)


affected_auc_df[
    "Difference_in_Full_Pair_Units"
] = (

    affected_auc_df[
        "Absolute_Difference"
    ]

    /

    fair_auc_pair_increment
)


affected_auc_df[
    "Difference_in_Half_Pair_Units"
] = (

    affected_auc_df[
        "Absolute_Difference"
    ]

    /

    fair_auc_half_pair_increment
)


display(
    affected_auc_df[
        [
            "Experiment",
            "Seed",
            "Phase4_Value",
            "Phase5_Reconstructed",
            "Absolute_Difference",
            "Difference_in_Full_Pair_Units",
            "Difference_in_Half_Pair_Units"
        ]
    ]
)

In [ ]:
# ============================================
# Section 5.13
# Save AUC Numerical Reproduction Audit
# ============================================

auc_reproduction_explanation_df = (
    affected_auc_df.copy()
)


auc_reproduction_explanation_df[
    "Interpretation"
] = (
    "Difference corresponds to <=1 positive-negative "
    "ranking-pair contribution in the Fair test set; "
    "classification-dependent metrics reproduce to "
    "floating-point precision."
)


auc_reproduction_explanation_df.to_csv(

    PHASE5_AUDIT_PATH
    / "Phase5_Corrected_AUC_Numerical_Reproduction_Audit.csv",

    index=False
)


print(
    "✓ AUC numerical reproduction audit saved."
)

## Section 5.14: Final Reproduction Audit Verdict

The corrected Phase 5 inference pipeline successfully reproduced the frozen corrected Phase 4 model behavior.

Across 315 metric comparisons, all threshold-dependent performance metrics reproduced to approximately machine precision.

Only six table entries exceeded \(10^{-10}\):

- three Fair AUC values,
- and the corresponding three AUC-gap values.

The maximum discrepancy was:

$$
1.6871\times10^{-5}.
$$

For the Fair test set:

$$
N_{+}=248,\qquad N_{-}=239,
$$

so a single positive-negative ranking comparison contributes:

$$
\frac{1}{248\times239}
\approx1.6871\times10^{-5}
$$

to ROC AUC.

The observed discrepancies therefore correspond to at most one ranking-pair contribution, with one case corresponding approximately to half of one pair contribution.

These extremely small differences are consistent with numerical ranking/tie sensitivity between inference executions and do not indicate a substantive mismatch in model reconstruction.

Accordingly:

> **The corrected Phase 5 checkpoint-to-prediction-to-metric pipeline is considered successfully reproduced and validated.**

The reconstructed Phase 5 predictions are therefore used for all subsequent mechanistic analyses.

# Section 6: Prediction Behavior and Predictive Uncertainty

This section evaluates whether controlled training-domain composition changes model prediction behavior across Fair and Dark skin-tone domains.

The primary prediction-level quantities are:

### Malignant probability

$$
P(Y=1)
$$

### Signed logit difference

$$
\Delta o=o_1-o_0
$$

where positive values favor the malignant class.

### Prediction confidence

$$
\max_k P(Y=k)
$$

### Predictive entropy

$$
H=-\sum_{k=0}^{1}P(Y=k)\log P(Y=k)
$$

For binary classification:

$$
0\le H\le\ln(2).
$$

Higher entropy indicates greater predictive uncertainty.

To avoid pseudoreplication, individual images are not treated as independent experimental replicates for inferential testing.

The independent unit remains the seed-specific trained model:

$$
n=5
$$

per training composition.

The predefined primary prediction-behavior hypothesis family contains four outcomes:

1. Fair–Dark malignant-probability difference for benign images,
2. Fair–Dark malignant-probability difference for malignant images,
3. Fair–Dark predictive-entropy difference for benign images,
4. Fair–Dark predictive-entropy difference for malignant images.

In [ ]:
# ============================================
# Section 6.1
# Add Human-Readable Class Labels
# ============================================

all_predictions_df[
    "Class"
] = (

    all_predictions_df[
        "True_Label"
    ]
    .map(
        {
            0: "Benign",
            1: "Malignant"
        }
    )
)


print(
    all_predictions_df[
        "Class"
    ]
    .value_counts()
)

In [ ]:
# ============================================
# Section 6.2
# Compute Model-Level Class-Conditional
# Prediction Behavior
# ============================================

behavior_rows = []


for experiment_name in EXPERIMENTS:

    for seed in SEEDS:

        for domain_name in [
            "Fair",
            "Dark"
        ]:

            for class_label, true_label in [
                ("Benign", 0),
                ("Malignant", 1)
            ]:

                subset = (

                    all_predictions_df[

                        (
                            all_predictions_df[
                                "Experiment"
                            ]
                            ==
                            experiment_name
                        )

                        &

                        (
                            all_predictions_df[
                                "Seed"
                            ]
                            ==
                            seed
                        )

                        &

                        (
                            all_predictions_df[
                                "Domain"
                            ]
                            ==
                            domain_name
                        )

                        &

                        (
                            all_predictions_df[
                                "True_Label"
                            ]
                            ==
                            true_label
                        )
                    ]
                )


                behavior_rows.append({

                    "Experiment":
                        experiment_name,

                    "Seed":
                        seed,

                    "Domain":
                        domain_name,

                    "Class":
                        class_label,

                    "N":
                        len(
                            subset
                        ),

                    "Mean_Malignant_Probability":
                        subset[
                            "Malignant_Probability"
                        ]
                        .mean(),

                    "Mean_Predictive_Entropy":
                        subset[
                            "Predictive_Entropy"
                        ]
                        .mean(),

                    "Mean_Confidence":
                        subset[
                            "Confidence"
                        ]
                        .mean(),

                    "Mean_Signed_Logit_Difference":
                        subset[
                            "Signed_Logit_Difference"
                        ]
                        .mean(),

                    "Mean_Absolute_Logit_Difference":
                        subset[
                            "Absolute_Logit_Difference"
                        ]
                        .mean()
                })


class_conditional_behavior_df = (
    pd.DataFrame(
        behavior_rows
    )
)


display(
    class_conditional_behavior_df.head(
        12
    )
)


print(
    "\nShape:",
    class_conditional_behavior_df.shape
)

In [ ]:
# ============================================
# Section 6.3
# Compute Within-Model Fair-Dark Differences
# ============================================

difference_rows = []


for experiment_name in EXPERIMENTS:

    for seed in SEEDS:

        for class_label in [
            "Benign",
            "Malignant"
        ]:

            subset = (

                class_conditional_behavior_df[

                    (
                        class_conditional_behavior_df[
                            "Experiment"
                        ]
                        ==
                        experiment_name
                    )

                    &

                    (
                        class_conditional_behavior_df[
                            "Seed"
                        ]
                        ==
                        seed
                    )

                    &

                    (
                        class_conditional_behavior_df[
                            "Class"
                        ]
                        ==
                        class_label
                    )
                ]
            )


            fair_row = (
                subset[
                    subset[
                        "Domain"
                    ]
                    ==
                    "Fair"
                ]
                .iloc[0]
            )


            dark_row = (
                subset[
                    subset[
                        "Domain"
                    ]
                    ==
                    "Dark"
                ]
                .iloc[0]
            )


            difference_rows.append({

                "Experiment":
                    experiment_name,

                "Seed":
                    seed,

                "Class":
                    class_label,

                "Malignant_Probability_Difference":
                    (
                        fair_row[
                            "Mean_Malignant_Probability"
                        ]
                        -
                        dark_row[
                            "Mean_Malignant_Probability"
                        ]
                    ),

                "Predictive_Entropy_Difference":
                    (
                        fair_row[
                            "Mean_Predictive_Entropy"
                        ]
                        -
                        dark_row[
                            "Mean_Predictive_Entropy"
                        ]
                    ),

                "Confidence_Difference":
                    (
                        fair_row[
                            "Mean_Confidence"
                        ]
                        -
                        dark_row[
                            "Mean_Confidence"
                        ]
                    ),

                "Signed_Logit_Difference_Domain_Gap":
                    (
                        fair_row[
                            "Mean_Signed_Logit_Difference"
                        ]
                        -
                        dark_row[
                            "Mean_Signed_Logit_Difference"
                        ]
                    )
            })


within_model_domain_difference_df = (
    pd.DataFrame(
        difference_rows
    )
)


display(
    within_model_domain_difference_df
)


print(
    "\nShape:",
    within_model_domain_difference_df.shape
)

In [ ]:
# ============================================
# Section 6.4
# Save Prediction-Behavior Tables
# ============================================

class_conditional_behavior_df.to_csv(

    PHASE5_TABLE_PATH
    / "Phase5_Corrected_Class_Conditional_Prediction_Behavior.csv",

    index=False
)


within_model_domain_difference_df.to_csv(

    PHASE5_TABLE_PATH
    / "Phase5_Corrected_Within_Model_Domain_Differences.csv",

    index=False
)


print(
    "✓ Prediction-behavior tables saved."
)

In [ ]:
# ============================================
# Section 6.5
# Summarize Fair-Dark Prediction Differences
# ============================================

behavior_summary_df = (

    within_model_domain_difference_df

    .groupby(
        [
            "Experiment",
            "Class"
        ]
    )

    .agg(

        Malignant_Probability_Difference_Mean=(
            "Malignant_Probability_Difference",
            "mean"
        ),

        Malignant_Probability_Difference_SD=(
            "Malignant_Probability_Difference",
            "std"
        ),

        Entropy_Difference_Mean=(
            "Predictive_Entropy_Difference",
            "mean"
        ),

        Entropy_Difference_SD=(
            "Predictive_Entropy_Difference",
            "std"
        )
    )

    .reset_index()
)


display(
    behavior_summary_df
)


behavior_summary_df.to_csv(

    PHASE5_TABLE_PATH
    / "Phase5_Corrected_Prediction_Behavior_Summary.csv",

    index=False
)

In [ ]:
# ============================================
# Section 6.6
# Exact Repeated-Measures Friedman Functions
# ============================================

from scipy.stats import rankdata
from statsmodels.stats.multitest import multipletests


def friedman_statistic(
    matrix
):

    matrix = np.asarray(
        matrix,
        dtype=float
    )

    n, k = matrix.shape


    ranked = np.apply_along_axis(
        rankdata,
        1,
        matrix
    )


    rank_sums = (
        ranked.sum(
            axis=0
        )
    )


    q = (

        12
        /
        (
            n
            *
            k
            *
            (k + 1)
        )

        *

        np.sum(
            rank_sums ** 2
        )

        -

        3
        *
        n
        *
        (k + 1)
    )


    return float(
        q
    )


def exact_friedman_test(
    matrix
):

    matrix = np.asarray(
        matrix,
        dtype=float
    )


    n, k = matrix.shape


    if k != 3:

        raise ValueError(
            "This exact implementation expects exactly 3 conditions."
        )


    observed_stat = (
        friedman_statistic(
            matrix
        )
    )


    condition_permutations = list(
        itertools.permutations(
            range(k)
        )
    )


    total_permutations = (
        len(
            condition_permutations
        )
        **
        n
    )


    extreme_count = 0


    for seed_permutations in itertools.product(

        condition_permutations,

        repeat=n
    ):

        permuted_matrix = np.empty_like(
            matrix
        )


        for row_index, permutation in enumerate(
            seed_permutations
        ):

            permuted_matrix[
                row_index
            ] = matrix[
                row_index,
                list(
                    permutation
                )
            ]


        permuted_stat = (
            friedman_statistic(
                permuted_matrix
            )
        )


        if (
            permuted_stat
            >=
            observed_stat
            -
            1e-12
        ):

            extreme_count += 1


    exact_p = (
        extreme_count
        /
        total_permutations
    )


    kendalls_w = (

        observed_stat
        /
        (
            n
            *
            (k - 1)
        )
    )


    return {

        "Friedman_Q":
            observed_stat,

        "Exact_P":
            exact_p,

        "Kendalls_W":
            kendalls_w,

        "Permutation_Count":
            total_permutations
    }

In [ ]:
# ============================================
# Section 6.7
# Define Predefined Prediction-Behavior Outcomes
# ============================================

PREDICTION_BEHAVIOR_OUTCOMES = [

    {
        "Outcome":
            "Malignant_Probability_Difference",

        "Class":
            "Benign",

        "Column":
            "Malignant_Probability_Difference"
    },

    {
        "Outcome":
            "Malignant_Probability_Difference",

        "Class":
            "Malignant",

        "Column":
            "Malignant_Probability_Difference"
    },

    {
        "Outcome":
            "Predictive_Entropy_Difference",

        "Class":
            "Benign",

        "Column":
            "Predictive_Entropy_Difference"
    },

    {
        "Outcome":
            "Predictive_Entropy_Difference",

        "Class":
            "Malignant",

        "Column":
            "Predictive_Entropy_Difference"
    }
]


EXPERIMENT_ORDER = [

    "M1_75F_25D",
    "M2_50F_50D",
    "M3_25F_75D"
]

In [ ]:
# ============================================
# Section 6.8
# Exact Friedman Tests
# Prediction-Behavior Family
# ============================================

behavior_stat_rows = []


for specification in (
    PREDICTION_BEHAVIOR_OUTCOMES
):

    class_label = (
        specification[
            "Class"
        ]
    )

    column_name = (
        specification[
            "Column"
        ]
    )


    class_df = (

        within_model_domain_difference_df[

            within_model_domain_difference_df[
                "Class"
            ]
            ==
            class_label
        ]

        .copy()
    )


    matrix_rows = []


    for seed in SEEDS:

        seed_values = []


        for experiment_name in (
            EXPERIMENT_ORDER
        ):

            value = float(

                class_df.loc[

                    (
                        class_df[
                            "Experiment"
                        ]
                        ==
                        experiment_name
                    )

                    &

                    (
                        class_df[
                            "Seed"
                        ]
                        ==
                        seed
                    ),

                    column_name
                ]
                .iloc[0]
            )


            seed_values.append(
                value
            )


        matrix_rows.append(
            seed_values
        )


    matched_matrix = np.asarray(
        matrix_rows,
        dtype=float
    )


    result = exact_friedman_test(
        matched_matrix
    )


    behavior_stat_rows.append({

        "Outcome":
            specification[
                "Outcome"
            ],

        "Class":
            class_label,

        **result
    })


prediction_behavior_friedman_df = (
    pd.DataFrame(
        behavior_stat_rows
    )
)


display(
    prediction_behavior_friedman_df
)

In [ ]:
# ============================================
# Section 6.9
# Holm Correction Across Prediction-Behavior Family
# ============================================

reject, p_holm, _, _ = multipletests(

    prediction_behavior_friedman_df[
        "Exact_P"
    ]
    .values,

    alpha=0.05,

    method="holm"
)


prediction_behavior_friedman_df[
    "Holm_P"
] = p_holm


prediction_behavior_friedman_df[
    "Supported_After_Holm"
] = reject


display(
    prediction_behavior_friedman_df
)

In [ ]:
# ============================================
# Section 6.10
# Save Prediction-Behavior Statistical Results
# ============================================

prediction_behavior_friedman_df.to_csv(

    PHASE5_TABLE_PATH
    / "Phase5_Corrected_Prediction_Behavior_Exact_Friedman.csv",

    index=False
)


print(
    "✓ Corrected prediction-behavior Friedman results saved."
)

In [ ]:
# ============================================
# Section 6.11
# Pairwise Exact Wilcoxon Tests
# Supported Malignant Entropy Outcome
# ============================================

from scipy.stats import wilcoxon


def exact_paired_wilcoxon(
    values_a,
    values_b
):

    values_a = np.asarray(
        values_a,
        dtype=float
    )

    values_b = np.asarray(
        values_b,
        dtype=float
    )


    differences = (
        values_a
        -
        values_b
    )


    nonzero_mask = (
        differences
        !=
        0
    )


    differences_nonzero = (
        differences[
            nonzero_mask
        ]
    )


    if len(
        differences_nonzero
    ) == 0:

        return {

            "Statistic":
                0.0,

            "Exact_P":
                1.0,

            "N_Nonzero":
                0,

            "Mean_Difference":
                0.0
        }


    result = wilcoxon(

        differences_nonzero,

        alternative="two-sided",

        zero_method="wilcox",

        method="exact"
    )


    return {

        "Statistic":
            float(
                result.statistic
            ),

        "Exact_P":
            float(
                result.pvalue
            ),

        "N_Nonzero":
            int(
                len(
                    differences_nonzero
                )
            ),

        "Mean_Difference":
            float(
                np.mean(
                    differences
                )
            )
    }

In [ ]:
# ============================================
# Section 6.12
# Extract Matched Malignant Entropy Differences
# ============================================

malignant_entropy_df = (

    within_model_domain_difference_df[

        within_model_domain_difference_df[
            "Class"
        ]
        ==
        "Malignant"
    ]

    [
        [
            "Experiment",
            "Seed",
            "Predictive_Entropy_Difference"
        ]
    ]

    .copy()
)


display(
    malignant_entropy_df
)


malignant_entropy_vectors = {}


for experiment_name in EXPERIMENT_ORDER:

    vector = (

        malignant_entropy_df[

            malignant_entropy_df[
                "Experiment"
            ]
            ==
            experiment_name
        ]

        .sort_values(
            "Seed"
        )

        [
            "Predictive_Entropy_Difference"
        ]

        .astype(float)

        .values
    )


    malignant_entropy_vectors[
        experiment_name
    ] = vector


    print(
        experiment_name,
        vector
    )

In [ ]:
# ============================================
# Section 6.13
# Pairwise Exact Wilcoxon Tests
# ============================================

pairwise_comparisons = [

    (
        "M1_75F_25D",
        "M2_50F_50D"
    ),

    (
        "M1_75F_25D",
        "M3_25F_75D"
    ),

    (
        "M2_50F_50D",
        "M3_25F_75D"
    )
]


malignant_entropy_pairwise_rows = []


for experiment_a, experiment_b in (
    pairwise_comparisons
):

    result = exact_paired_wilcoxon(

        malignant_entropy_vectors[
            experiment_a
        ],

        malignant_entropy_vectors[
            experiment_b
        ]
    )


    malignant_entropy_pairwise_rows.append({

        "Comparison":
            f"{experiment_a} vs {experiment_b}",

        **result
    })


malignant_entropy_pairwise_df = (
    pd.DataFrame(
        malignant_entropy_pairwise_rows
    )
)


display(
    malignant_entropy_pairwise_df
)

In [ ]:
# ============================================
# Section 6.14
# Holm Correction Across Three Post-Hoc Tests
# ============================================

reject, p_holm, _, _ = multipletests(

    malignant_entropy_pairwise_df[
        "Exact_P"
    ]
    .values,

    alpha=0.05,

    method="holm"
)


malignant_entropy_pairwise_df[
    "Holm_P"
] = p_holm


malignant_entropy_pairwise_df[
    "Supported_After_Holm"
] = reject


display(
    malignant_entropy_pairwise_df
)

In [ ]:
# ============================================
# Section 6.15
# Bootstrap Paired Mean Differences
# Malignant Entropy
# ============================================

BOOTSTRAP_ITERATIONS = 10000
STAT_SEED = 42


rng = np.random.default_rng(
    STAT_SEED
)


bootstrap_rows = []


for experiment_a, experiment_b in (
    pairwise_comparisons
):

    values_a = (
        malignant_entropy_vectors[
            experiment_a
        ]
    )

    values_b = (
        malignant_entropy_vectors[
            experiment_b
        ]
    )


    paired_differences = (
        values_a
        -
        values_b
    )


    bootstrap_means = []


    for _ in range(
        BOOTSTRAP_ITERATIONS
    ):

        sampled_indices = rng.integers(

            low=0,

            high=len(
                paired_differences
            ),

            size=len(
                paired_differences
            )
        )


        bootstrap_means.append(

            paired_differences[
                sampled_indices
            ]
            .mean()
        )


    lower_ci = np.percentile(
        bootstrap_means,
        2.5
    )

    upper_ci = np.percentile(
        bootstrap_means,
        97.5
    )


    bootstrap_rows.append({

        "Comparison":
            f"{experiment_a} vs {experiment_b}",

        "Mean_Difference":
            float(
                paired_differences.mean()
            ),

        "Bootstrap_95CI_Lower":
            float(
                lower_ci
            ),

        "Bootstrap_95CI_Upper":
            float(
                upper_ci
            ),

        "Bootstrap_Iterations":
            BOOTSTRAP_ITERATIONS
    })


malignant_entropy_bootstrap_df = (
    pd.DataFrame(
        bootstrap_rows
    )
)


display(
    malignant_entropy_bootstrap_df
)

In [ ]:
# ============================================
# Section 6.16
# Save Malignant Entropy Follow-Up Results
# ============================================

malignant_entropy_pairwise_df.to_csv(

    PHASE5_TABLE_PATH
    / "Phase5_Corrected_Malignant_Entropy_Posthoc.csv",

    index=False
)


malignant_entropy_bootstrap_df.to_csv(

    PHASE5_TABLE_PATH
    / "Phase5_Corrected_Malignant_Entropy_Bootstrap.csv",

    index=False
)


print(
    "✓ Malignant entropy follow-up results saved."
)

## Section 6.17: Confirmatory Prediction-Behavior Result

Among the four predefined prediction-behavior outcomes, only the Fair–Dark predictive-entropy difference for malignant images remained statistically supported after Holm correction.

The exact repeated-measures result was:

$$
Q=10.0
$$

$$
p_{\mathrm{exact}}=0.000772
$$

$$
W=1.00
$$

and:

$$
p_{\mathrm{Holm}}=0.003086.
$$

Thus, controlled training-domain composition produced a statistically supported change in the Fair–Dark predictive-uncertainty disparity for malignant lesions.

The remaining three predefined prediction-behavior outcomes did not remain statistically supported after multiplicity correction.

Any pairwise Wilcoxon follow-up results are interpreted within the limits imposed by only five paired seed-level observations.

# Section 7: Calibration and Probabilistic Error

This section evaluates whether Fair and Dark test domains differ in probabilistic prediction error.

The primary calibration-related measure is the Brier score.

For binary classification:

$$
BS=
\frac{1}{N}
\sum_{i=1}^{N}
(p_i-y_i)^2
$$

where:

- $p_i$ is predicted malignant probability,
- $y_i\in\{0,1\}$ is the true class.

Lower Brier score indicates better probabilistic accuracy.

Because Fair and Dark test sets differ in disease composition, the primary inferential analysis is class-conditional.

Brier error is therefore evaluated separately for:

1. benign images,
2. malignant images.

The independent experimental unit remains the seed-specific trained model.

In [ ]:
# ============================================
# Section 7.1
# Add Image-Level Brier Error
# ============================================

all_predictions_df[
    "Brier_Error"
] = (

    all_predictions_df[
        "Malignant_Probability"
    ]

    -

    all_predictions_df[
        "True_Label"
    ]

) ** 2


print(
    "Brier error range:"
)

print(
    all_predictions_df[
        "Brier_Error"
    ].min(),
    all_predictions_df[
        "Brier_Error"
    ].max()
)

In [ ]:
# ============================================
# Section 7.2
# Compute Model-Level Domain Brier Scores
# ============================================

calibration_rows = []


for experiment_name in EXPERIMENTS:

    for seed in SEEDS:

        for domain_name in [
            "Fair",
            "Dark"
        ]:

            subset = (

                all_predictions_df[

                    (
                        all_predictions_df[
                            "Experiment"
                        ]
                        ==
                        experiment_name
                    )

                    &

                    (
                        all_predictions_df[
                            "Seed"
                        ]
                        ==
                        seed
                    )

                    &

                    (
                        all_predictions_df[
                            "Domain"
                        ]
                        ==
                        domain_name
                    )
                ]
            )


            calibration_rows.append({

                "Experiment":
                    experiment_name,

                "Seed":
                    seed,

                "Domain":
                    domain_name,

                "N":
                    len(
                        subset
                    ),

                "Brier_Score":
                    subset[
                        "Brier_Error"
                    ]
                    .mean()
            })


model_level_calibration_df = (
    pd.DataFrame(
        calibration_rows
    )
)


display(
    model_level_calibration_df.head(
        10
    )
)

In [ ]:
# ============================================
# Section 7.3
# Compute Model-Level Fair-Dark
# Brier Differences
# ============================================

calibration_difference_rows = []


for experiment_name in EXPERIMENTS:

    for seed in SEEDS:

        run_df = (

            model_level_calibration_df[

                (
                    model_level_calibration_df[
                        "Experiment"
                    ]
                    ==
                    experiment_name
                )

                &

                (
                    model_level_calibration_df[
                        "Seed"
                    ]
                    ==
                    seed
                )
            ]
        )


        fair_brier = float(

            run_df.loc[

                run_df[
                    "Domain"
                ]
                ==
                "Fair",

                "Brier_Score"
            ]
            .iloc[0]
        )


        dark_brier = float(

            run_df.loc[

                run_df[
                    "Domain"
                ]
                ==
                "Dark",

                "Brier_Score"
            ]
            .iloc[0]
        )


        calibration_difference_rows.append({

            "Experiment":
                experiment_name,

            "Seed":
                seed,

            "Fair_Brier":
                fair_brier,

            "Dark_Brier":
                dark_brier,

            "Brier_Difference":
                (
                    fair_brier
                    -
                    dark_brier
                ),

            "Absolute_Brier_Gap":
                abs(
                    fair_brier
                    -
                    dark_brier
                )
        })


calibration_domain_difference_df = (
    pd.DataFrame(
        calibration_difference_rows
    )
)


display(
    calibration_domain_difference_df
)

In [ ]:
# ============================================
# Section 7.4
# Compute Class-Conditional Brier Scores
# ============================================

class_brier_rows = []


for experiment_name in EXPERIMENTS:

    for seed in SEEDS:

        for domain_name in [
            "Fair",
            "Dark"
        ]:

            for class_label, true_label in [
                ("Benign", 0),
                ("Malignant", 1)
            ]:

                subset = (

                    all_predictions_df[

                        (
                            all_predictions_df[
                                "Experiment"
                            ]
                            ==
                            experiment_name
                        )

                        &

                        (
                            all_predictions_df[
                                "Seed"
                            ]
                            ==
                            seed
                        )

                        &

                        (
                            all_predictions_df[
                                "Domain"
                            ]
                            ==
                            domain_name
                        )

                        &

                        (
                            all_predictions_df[
                                "True_Label"
                            ]
                            ==
                            true_label
                        )
                    ]
                )


                class_brier_rows.append({

                    "Experiment":
                        experiment_name,

                    "Seed":
                        seed,

                    "Domain":
                        domain_name,

                    "Class":
                        class_label,

                    "N":
                        len(
                            subset
                        ),

                    "Brier_Score":
                        subset[
                            "Brier_Error"
                        ]
                        .mean()
                })


class_conditional_brier_df = (
    pd.DataFrame(
        class_brier_rows
    )
)


display(
    class_conditional_brier_df.head(
        12
    )
)


print(
    "\nShape:",
    class_conditional_brier_df.shape
)

In [ ]:
# ============================================
# Section 7.5
# Compute Class-Conditional Fair-Dark
# Brier Differences
# ============================================

class_brier_difference_rows = []


for experiment_name in EXPERIMENTS:

    for seed in SEEDS:

        for class_label in [
            "Benign",
            "Malignant"
        ]:

            subset = (

                class_conditional_brier_df[

                    (
                        class_conditional_brier_df[
                            "Experiment"
                        ]
                        ==
                        experiment_name
                    )

                    &

                    (
                        class_conditional_brier_df[
                            "Seed"
                        ]
                        ==
                        seed
                    )

                    &

                    (
                        class_conditional_brier_df[
                            "Class"
                        ]
                        ==
                        class_label
                    )
                ]
            )


            fair_brier = float(

                subset.loc[

                    subset[
                        "Domain"
                    ]
                    ==
                    "Fair",

                    "Brier_Score"
                ]
                .iloc[0]
            )


            dark_brier = float(

                subset.loc[

                    subset[
                        "Domain"
                    ]
                    ==
                    "Dark",

                    "Brier_Score"
                ]
                .iloc[0]
            )


            class_brier_difference_rows.append({

                "Experiment":
                    experiment_name,

                "Seed":
                    seed,

                "Class":
                    class_label,

                "Fair_Brier":
                    fair_brier,

                "Dark_Brier":
                    dark_brier,

                "Brier_Difference":
                    (
                        fair_brier
                        -
                        dark_brier
                    )
            })


class_brier_difference_df = (
    pd.DataFrame(
        class_brier_difference_rows
    )
)


display(
    class_brier_difference_df
)

In [ ]:
# ============================================
# Section 7.6
# Summarize Class-Conditional Brier Differences
# ============================================

class_brier_summary_df = (

    class_brier_difference_df

    .groupby(
        [
            "Experiment",
            "Class"
        ]
    )

    .agg(

        Fair_Brier_Mean=(
            "Fair_Brier",
            "mean"
        ),

        Fair_Brier_SD=(
            "Fair_Brier",
            "std"
        ),

        Dark_Brier_Mean=(
            "Dark_Brier",
            "mean"
        ),

        Dark_Brier_SD=(
            "Dark_Brier",
            "std"
        ),

        Difference_Mean=(
            "Brier_Difference",
            "mean"
        ),

        Difference_SD=(
            "Brier_Difference",
            "std"
        )
    )

    .reset_index()
)


display(
    class_brier_summary_df
)

In [ ]:
# ============================================
# Section 7.7
# Exact Friedman Tests
# Class-Conditional Brier Differences
# ============================================

brier_stat_rows = []


for class_label in [
    "Benign",
    "Malignant"
]:

    class_df = (

        class_brier_difference_df[

            class_brier_difference_df[
                "Class"
            ]
            ==
            class_label
        ]
    )


    matrix_rows = []


    for seed in SEEDS:

        seed_values = []


        for experiment_name in (
            EXPERIMENT_ORDER
        ):

            value = float(

                class_df.loc[

                    (
                        class_df[
                            "Experiment"
                        ]
                        ==
                        experiment_name
                    )

                    &

                    (
                        class_df[
                            "Seed"
                        ]
                        ==
                        seed
                    ),

                    "Brier_Difference"
                ]
                .iloc[0]
            )


            seed_values.append(
                value
            )


        matrix_rows.append(
            seed_values
        )


    matrix = np.asarray(
        matrix_rows,
        dtype=float
    )


    result = exact_friedman_test(
        matrix
    )


    brier_stat_rows.append({

        "Outcome":
            "Brier_Difference",

        "Class":
            class_label,

        **result
    })


class_brier_friedman_df = (
    pd.DataFrame(
        brier_stat_rows
    )
)


display(
    class_brier_friedman_df
)

In [ ]:
# ============================================
# Section 7.8
# Holm Correction Across Brier Hypotheses
# ============================================

reject, p_holm, _, _ = multipletests(

    class_brier_friedman_df[
        "Exact_P"
    ]
    .values,

    alpha=0.05,

    method="holm"
)


class_brier_friedman_df[
    "Holm_P"
] = p_holm


class_brier_friedman_df[
    "Supported_After_Holm"
] = reject


display(
    class_brier_friedman_df
)

In [ ]:
# ============================================
# Section 7.9
# Save Calibration and Brier Artifacts
# ============================================

model_level_calibration_df.to_csv(

    PHASE5_TABLE_PATH
    / "Phase5_Corrected_Model_Level_Calibration.csv",

    index=False
)


calibration_domain_difference_df.to_csv(

    PHASE5_TABLE_PATH
    / "Phase5_Corrected_Calibration_Domain_Differences.csv",

    index=False
)


class_conditional_brier_df.to_csv(

    PHASE5_TABLE_PATH
    / "Phase5_Corrected_Class_Conditional_Brier.csv",

    index=False
)


class_brier_summary_df.to_csv(

    PHASE5_TABLE_PATH
    / "Phase5_Corrected_Class_Conditional_Brier_Summary.csv",

    index=False
)


class_brier_friedman_df.to_csv(

    PHASE5_TABLE_PATH
    / "Phase5_Corrected_Class_Conditional_Brier_Exact_Friedman.csv",

    index=False
)


print(
    "✓ Corrected calibration/Brier artifacts saved."
)

In [ ]:
# ============================================
# Section 7.10
# Extract Matched Malignant Brier Differences
# ============================================

malignant_brier_df = (

    class_brier_difference_df[

        class_brier_difference_df[
            "Class"
        ]
        ==
        "Malignant"
    ]

    [
        [
            "Experiment",
            "Seed",
            "Brier_Difference"
        ]
    ]

    .copy()
)


display(
    malignant_brier_df
)


malignant_brier_vectors = {}


for experiment_name in EXPERIMENT_ORDER:

    vector = (

        malignant_brier_df[

            malignant_brier_df[
                "Experiment"
            ]
            ==
            experiment_name
        ]

        .sort_values(
            "Seed"
        )

        [
            "Brier_Difference"
        ]

        .astype(float)

        .values
    )


    malignant_brier_vectors[
        experiment_name
    ] = vector


    print(
        experiment_name,
        vector
    )

In [ ]:
# ============================================
# Section 7.11
# Pairwise Exact Wilcoxon Tests
# Malignant Brier Difference
# ============================================

malignant_brier_pairwise_rows = []


for experiment_a, experiment_b in (
    pairwise_comparisons
):

    result = exact_paired_wilcoxon(

        malignant_brier_vectors[
            experiment_a
        ],

        malignant_brier_vectors[
            experiment_b
        ]
    )


    malignant_brier_pairwise_rows.append({

        "Comparison":
            f"{experiment_a} vs {experiment_b}",

        **result
    })


malignant_brier_pairwise_df = (
    pd.DataFrame(
        malignant_brier_pairwise_rows
    )
)


display(
    malignant_brier_pairwise_df
)

In [ ]:
# ============================================
# Section 7.12
# Holm Correction Across Malignant Brier
# Pairwise Comparisons
# ============================================

reject, p_holm, _, _ = multipletests(

    malignant_brier_pairwise_df[
        "Exact_P"
    ]
    .values,

    alpha=0.05,

    method="holm"
)


malignant_brier_pairwise_df[
    "Holm_P"
] = p_holm


malignant_brier_pairwise_df[
    "Supported_After_Holm"
] = reject


display(
    malignant_brier_pairwise_df
)

In [ ]:
# ============================================
# Section 7.13
# Bootstrap Paired Mean Differences
# Malignant Brier
# ============================================

rng = np.random.default_rng(
    42
)


malignant_brier_bootstrap_rows = []


for experiment_a, experiment_b in (
    pairwise_comparisons
):

    values_a = (
        malignant_brier_vectors[
            experiment_a
        ]
    )

    values_b = (
        malignant_brier_vectors[
            experiment_b
        ]
    )


    paired_differences = (
        values_a
        -
        values_b
    )


    bootstrap_means = []


    for _ in range(
        10000
    ):

        sampled_indices = rng.integers(

            0,

            len(
                paired_differences
            ),

            size=len(
                paired_differences
            )
        )


        bootstrap_means.append(

            paired_differences[
                sampled_indices
            ]
            .mean()
        )


    malignant_brier_bootstrap_rows.append({

        "Comparison":
            f"{experiment_a} vs {experiment_b}",

        "Mean_Difference":
            float(
                paired_differences.mean()
            ),

        "Bootstrap_95CI_Lower":
            float(
                np.percentile(
                    bootstrap_means,
                    2.5
                )
            ),

        "Bootstrap_95CI_Upper":
            float(
                np.percentile(
                    bootstrap_means,
                    97.5
                )
            ),

        "Bootstrap_Iterations":
            10000
    })


malignant_brier_bootstrap_df = (
    pd.DataFrame(
        malignant_brier_bootstrap_rows
    )
)


display(
    malignant_brier_bootstrap_df
)

In [ ]:
# ============================================
# Section 7.14
# Save Malignant Brier Follow-Up Results
# ============================================

malignant_brier_pairwise_df.to_csv(

    PHASE5_TABLE_PATH
    / "Phase5_Corrected_Malignant_Brier_Posthoc.csv",

    index=False
)


malignant_brier_bootstrap_df.to_csv(

    PHASE5_TABLE_PATH
    / "Phase5_Corrected_Malignant_Brier_Bootstrap.csv",

    index=False
)


print(
    "✓ Malignant Brier follow-up artifacts saved."
)

## Section 7.15: Calibration Result

The Fair–Dark class-conditional Brier disparity for malignant lesions changed significantly across controlled training compositions.

The exact repeated-measures result was:

$$
Q=8.4
$$

$$
p_{\mathrm{exact}}=0.008488
$$

$$
W=0.84
$$

with Holm-adjusted:

$$
p_{\mathrm{Holm}}=0.016975.
$$

Therefore, training composition affected malignant probabilistic prediction error differently across Fair and Dark domains.

The corresponding benign Brier disparity was not statistically supported after multiplicity correction.

The malignant Brier pattern is not interpreted as a simple monotonic improvement with increasing Dark representation; rather, training composition altered the relative probabilistic error structure between domains.

# Section 8: Malignant False-Negative Persistence and Hard-Case Structure

This section investigates whether malignant classification failures are concentrated in a stable subset of test images.

For malignant images:

$$
Y=1
$$

a false negative occurs when:

$$
\hat{Y}=0.
$$

For each malignant test image, false-negative frequency is calculated across the five independently trained seeds within each training composition.

Thus, for a given image and composition:

$$
FN_{\mathrm{count}}\in\{0,1,2,3,4,5\}.
$$

A value of:

$$
FN_{\mathrm{count}}=5
$$

means the same malignant image was misclassified by every seed-specific model within that composition.

This section is primarily descriptive and examines:

- false-negative persistence,
- changes in persistence across compositions,
- overlap of repeatedly difficult malignant cases.

Individual images are not treated as independent model replications for confirmatory inference.

In [ ]:
# ============================================
# Section 8.1
# Extract Malignant Test Predictions
# ============================================

malignant_predictions_df = (

    all_predictions_df[

        all_predictions_df[
            "True_Label"
        ]
        ==
        1
    ]

    .copy()
)


malignant_predictions_df[
    "False_Negative"
] = (

    malignant_predictions_df[
        "Predicted_Label"
    ]
    ==
    0

).astype(int)


print(
    "Malignant prediction records:",
    len(
        malignant_predictions_df
    )
)


print(
    "\nBy domain:"
)

print(
    malignant_predictions_df[
        "Domain"
    ]
    .value_counts()
)


print(
    "\nTotal false-negative predictions:",
    malignant_predictions_df[
        "False_Negative"
    ]
    .sum()
)

In [ ]:
# ============================================
# Section 8.2
# Compute Image-Level False-Negative Persistence
# ============================================

fn_persistence_df = (

    malignant_predictions_df

    .groupby(
        [
            "Experiment",
            "Domain",
            "md5hash",
            "image_path"
        ],
        as_index=False
    )

    .agg(

        Seed_Count=(
            "Seed",
            "nunique"
        ),

        False_Negative_Count=(
            "False_Negative",
            "sum"
        ),

        Mean_Malignant_Probability=(
            "Malignant_Probability",
            "mean"
        ),

        SD_Malignant_Probability=(
            "Malignant_Probability",
            "std"
        ),

        Mean_Predictive_Entropy=(
            "Predictive_Entropy",
            "mean"
        )
    )
)


fn_persistence_df[
    "False_Negative_Rate_Across_Seeds"
] = (

    fn_persistence_df[
        "False_Negative_Count"
    ]

    /

    fn_persistence_df[
        "Seed_Count"
    ]
)


display(
    fn_persistence_df.head()
)


print(
    "\nSeed-count values:"
)

print(
    fn_persistence_df[
        "Seed_Count"
    ]
    .value_counts()
    .sort_index()
)

In [ ]:
# ============================================
# Section 8.3
# Summarize False-Negative Persistence
# ============================================

fn_persistence_summary_df = (

    fn_persistence_df

    .groupby(
        [
            "Experiment",
            "Domain",
            "False_Negative_Count"
        ]
    )

    .size()

    .reset_index(
        name="Image_Count"
    )
)


display(
    fn_persistence_summary_df
)

In [ ]:
# ============================================
# Section 8.4
# Identify Persistent Malignant Hard Cases
# ============================================

HARD_CASE_THRESHOLD = 4


persistent_hard_cases_df = (

    fn_persistence_df[

        fn_persistence_df[
            "False_Negative_Count"
        ]
        >=
        HARD_CASE_THRESHOLD
    ]

    .copy()
)


hard_case_summary_df = (

    persistent_hard_cases_df

    .groupby(
        [
            "Experiment",
            "Domain"
        ]
    )

    .agg(

        Persistent_Hard_Cases=(
            "md5hash",
            "nunique"
        ),

        Mean_FN_Count=(
            "False_Negative_Count",
            "mean"
        ),

        Mean_Malignant_Probability=(
            "Mean_Malignant_Probability",
            "mean"
        )
    )

    .reset_index()
)


display(
    hard_case_summary_df
)

In [ ]:
# ============================================
# Section 8.5
# Dark Malignant Persistence Summary
# ============================================

dark_fn_persistence_df = (

    fn_persistence_df[

        fn_persistence_df[
            "Domain"
        ]
        ==
        "Dark"
    ]

    .copy()
)


dark_persistence_summary_df = (

    dark_fn_persistence_df

    .groupby(
        "Experiment"
    )

    .agg(

        Total_Malignant_Images=(
            "md5hash",
            "nunique"
        ),

        Mean_FN_Count=(
            "False_Negative_Count",
            "mean"
        ),

        Median_FN_Count=(
            "False_Negative_Count",
            "median"
        ),

        Images_Missed_At_Least_Once=(
            "False_Negative_Count",
            lambda x: int(
                (x >= 1).sum()
            )
        ),

        Images_Missed_At_Least_3_Seeds=(
            "False_Negative_Count",
            lambda x: int(
                (x >= 3).sum()
            )
        ),

        Persistent_Hard_Cases_4plus=(
            "False_Negative_Count",
            lambda x: int(
                (x >= 4).sum()
            )
        ),

        Missed_By_All_5=(
            "False_Negative_Count",
            lambda x: int(
                (x == 5).sum()
            )
        )
    )

    .reset_index()
)


display(
    dark_persistence_summary_df
)

In [ ]:
# ============================================
# Section 8.6
# Create Dark Malignant Cross-Composition
# Persistence Matrix
# ============================================

dark_persistence_matrix_df = (

    dark_fn_persistence_df

    .pivot_table(

        index=[
            "md5hash",
            "image_path"
        ],

        columns="Experiment",

        values="False_Negative_Count"
    )

    .reset_index()
)


dark_persistence_matrix_df.columns.name = None


display(
    dark_persistence_matrix_df.head()
)


print(
    "Dark malignant images:",
    len(
        dark_persistence_matrix_df
    )
)

In [ ]:
# ============================================
# Section 8.7
# Quantify Persistence Changes
# Across Compositions
# ============================================

dark_persistence_matrix_df[
    "M2_minus_M1"
] = (

    dark_persistence_matrix_df[
        "M2_50F_50D"
    ]

    -

    dark_persistence_matrix_df[
        "M1_75F_25D"
    ]
)


dark_persistence_matrix_df[
    "M3_minus_M1"
] = (

    dark_persistence_matrix_df[
        "M3_25F_75D"
    ]

    -

    dark_persistence_matrix_df[
        "M1_75F_25D"
    ]
)


dark_persistence_matrix_df[
    "M3_minus_M2"
] = (

    dark_persistence_matrix_df[
        "M3_25F_75D"
    ]

    -

    dark_persistence_matrix_df[
        "M2_50F_50D"
    ]
)


change_summary_rows = []


for column_name in [

    "M2_minus_M1",
    "M3_minus_M1",
    "M3_minus_M2"

]:

    values = (
        dark_persistence_matrix_df[
            column_name
        ]
    )


    change_summary_rows.append({

        "Comparison":
            column_name,

        "Mean_Change":
            values.mean(),

        "Median_Change":
            values.median(),

        "Improved_Images":
            int(
                (values < 0).sum()
            ),

        "Unchanged_Images":
            int(
                (values == 0).sum()
            ),

        "Worsened_Images":
            int(
                (values > 0).sum()
            )
    })


malignant_error_changes_df = (
    pd.DataFrame(
        change_summary_rows
    )
)


display(
    malignant_error_changes_df
)

In [ ]:
# ============================================
# Section 8.8
# Correlate Image-Level Difficulty
# Across Compositions
# ============================================

from scipy.stats import spearmanr


correlation_pairs = [

    (
        "M1_75F_25D",
        "M2_50F_50D"
    ),

    (
        "M1_75F_25D",
        "M3_25F_75D"
    ),

    (
        "M2_50F_50D",
        "M3_25F_75D"
    )
]


correlation_rows = []


for composition_a, composition_b in (
    correlation_pairs
):

    rho, p_value = spearmanr(

        dark_persistence_matrix_df[
            composition_a
        ],

        dark_persistence_matrix_df[
            composition_b
        ]
    )


    correlation_rows.append({

        "Comparison":
            f"{composition_a} vs {composition_b}",

        "Spearman_Rho":
            rho,

        "P_Value":
            p_value
    })


malignant_error_correlation_df = (
    pd.DataFrame(
        correlation_rows
    )
)


display(
    malignant_error_correlation_df
)

In [ ]:
# ============================================
# Section 8.9
# Persistent Hard-Case Overlap
# Across Compositions
# ============================================

hard_case_sets = {}


for experiment_name in (
    EXPERIMENT_ORDER
):

    hard_case_sets[
        experiment_name
    ] = set(

        dark_fn_persistence_df.loc[

            (
                dark_fn_persistence_df[
                    "Experiment"
                ]
                ==
                experiment_name
            )

            &

            (
                dark_fn_persistence_df[
                    "False_Negative_Count"
                ]
                >=
                HARD_CASE_THRESHOLD
            ),

            "md5hash"
        ]
    )


hard_overlap_rows = []


for composition_a, composition_b in (
    correlation_pairs
):

    set_a = (
        hard_case_sets[
            composition_a
        ]
    )

    set_b = (
        hard_case_sets[
            composition_b
        ]
    )


    intersection = (
        set_a
        &
        set_b
    )


    union = (
        set_a
        |
        set_b
    )


    jaccard = (

        len(
            intersection
        )
        /
        len(
            union
        )

        if len(
            union
        )
        >
        0

        else np.nan
    )


    hard_overlap_rows.append({

        "Comparison":
            f"{composition_a} vs {composition_b}",

        "Hard_Cases_A":
            len(
                set_a
            ),

        "Hard_Cases_B":
            len(
                set_b
            ),

        "Shared_Hard_Cases":
            len(
                intersection
            ),

        "Union_Hard_Cases":
            len(
                union
            ),

        "Jaccard_Overlap":
            jaccard
    })


malignant_hard_case_overlap_df = (
    pd.DataFrame(
        hard_overlap_rows
    )
)


display(
    malignant_hard_case_overlap_df
)

In [ ]:
# ============================================
# Section 8.10
# Save Malignant Error-Persistence Artifacts
# ============================================

fn_persistence_df.to_csv(

    PHASE5_TABLE_PATH
    / "Phase5_Corrected_Malignant_Error_Persistence.csv",

    index=False
)


dark_persistence_matrix_df.to_csv(

    PHASE5_TABLE_PATH
    / "Phase5_Corrected_Dark_Malignant_Persistence_Matrix.csv",

    index=False
)


malignant_error_changes_df.to_csv(

    PHASE5_TABLE_PATH
    / "Phase5_Corrected_Malignant_Error_Changes.csv",

    index=False
)


malignant_error_correlation_df.to_csv(

    PHASE5_TABLE_PATH
    / "Phase5_Corrected_Malignant_Error_Persistence_Correlation.csv",

    index=False
)


malignant_hard_case_overlap_df.to_csv(

    PHASE5_TABLE_PATH
    / "Phase5_Corrected_Malignant_Hard_Case_Overlap.csv",

    index=False
)


print(
    "✓ Malignant error-persistence artifacts saved."
)

# Section 9: Learned Avgpool Representation Extraction

This section analyzes the internal representations learned by the corrected Phase 4 ResNet50 models.

The final convolutional representation immediately before the classification layer is obtained from the ResNet50 average-pooling layer.

For every test image:

$$
z\in\mathbb{R}^{2048}
$$

is extracted.

Representations are retained separately for every:

- training composition,
- seed,
- Fair/Dark domain,
- disease class,
- test image.

No model parameters are changed.

The purpose is to evaluate whether controlled Fair/Dark training composition changes the learned feature-distribution discrepancy between Fair and Dark skin-tone domains.

Because independently trained neural networks can use different internal coordinate systems, raw feature vectors from different models are not pooled into one shared representation space for PCA.

Each seed-specific model remains an independent experimental replicate.

In [ ]:
# ============================================
# Section 9.0
# Runtime Recovery Guard
# ============================================

from pathlib import Path

import os
import json
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image


ARTIFACT_PATH = Path(
    "/content/drive/MyDrive/Phase_3_Artifacts"
)

DATA_PATH = (
    ARTIFACT_PATH
    / "dataset_splits"
)

CORRECTED_PHASE4_PATH = (
    ARTIFACT_PATH
    / "Phase_4_Mixing_Experiments_Corrected"
)

CORRECTED_PHASE4_MODEL_PATH = (
    CORRECTED_PHASE4_PATH
    / "models"
)

CORRECTED_PHASE5_PATH = (
    ARTIFACT_PATH
    / "Phase_5_Mechanistic_Analysis_Corrected"
)

PHASE5_REPRESENTATION_PATH = (
    CORRECTED_PHASE5_PATH
    / "representations"
)

PHASE5_TABLE_PATH = (
    CORRECTED_PHASE5_PATH
    / "tables"
)

PHASE5_REPRESENTATION_PATH.mkdir(
    parents=True,
    exist_ok=True
)

PHASE5_TABLE_PATH.mkdir(
    parents=True,
    exist_ok=True
)


DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


EXPERIMENTS = [
    "M1_75F_25D",
    "M2_50F_50D",
    "M3_25F_75D"
]

EXPERIMENT_ORDER = EXPERIMENTS.copy()

SEEDS = [
    42,
    43,
    44,
    45,
    46
]


print(
    "Device:",
    DEVICE
)

print(
    "Representation path:",
    PHASE5_REPRESENTATION_PATH
)

In [ ]:
# ============================================
# Section 9.1
# Restore Frozen Test Data and Transform
# ============================================

fair_test_df = pd.read_csv(
    DATA_PATH
    / "fair_test.csv"
)

dark_test_df = pd.read_csv(
    DATA_PATH
    / "dark_test.csv"
)


with open(
    ARTIFACT_PATH
    / "preprocessing_config.json",
    "r"
) as f:

    preprocessing_config = json.load(f)


IMAGE_SIZE = (
    preprocessing_config[
        "image_size"
    ]
)

NORMALIZE_MEAN = (
    preprocessing_config[
        "normalization"
    ]["mean"]
)

NORMALIZE_STD = (
    preprocessing_config[
        "normalization"
    ]["std"]
)


eval_transform = transforms.Compose([

    transforms.Resize(
        (
            IMAGE_SIZE,
            IMAGE_SIZE
        )
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=NORMALIZE_MEAN,
        std=NORMALIZE_STD
    )
])


class ClassificationDataset(
    Dataset
):

    def __init__(
        self,
        dataframe,
        transform=None
    ):

        self.dataframe = (
            dataframe
            .reset_index(
                drop=True
            )
        )

        self.transform = transform


    def __len__(
        self
    ):

        return len(
            self.dataframe
        )


    def __getitem__(
        self,
        idx
    ):

        row = (
            self.dataframe
            .iloc[idx]
        )


        image = Image.open(
            row[
                "image_path"
            ]
        ).convert(
            "RGB"
        )


        if self.transform:

            image = self.transform(
                image
            )


        label = torch.tensor(
            int(
                row[
                    "target"
                ]
            ),
            dtype=torch.long
        )


        return (
            image,
            label,
            idx
        )


fair_test_dataset = ClassificationDataset(
    fair_test_df,
    eval_transform
)

dark_test_dataset = ClassificationDataset(
    dark_test_df,
    eval_transform
)


fair_test_loader = DataLoader(

    fair_test_dataset,

    batch_size=32,

    shuffle=False,

    num_workers=0,

    pin_memory=torch.cuda.is_available()
)


dark_test_loader = DataLoader(

    dark_test_dataset,

    batch_size=32,

    shuffle=False,

    num_workers=0,

    pin_memory=torch.cuda.is_available()
)


print(
    "Fair:",
    len(
        fair_test_dataset
    )
)

print(
    "Dark:",
    len(
        dark_test_dataset
    )
)

In [ ]:
# ============================================
# Section 9.2
# Define Frozen ResNet50 Architecture
# ============================================

NUM_CLASSES = 2


def create_phase4_resnet50():

    model = models.resnet50(
        weights=None
    )


    num_features = (
        model.fc
        .in_features
    )


    model.fc = nn.Linear(
        num_features,
        NUM_CLASSES
    )


    return model.to(
        DEVICE
    )


print(
    "✓ Model reconstruction function ready."
)

In [ ]:
# ============================================
# Section 9.3
# Define Avgpool Representation Extraction
# ============================================

def extract_avgpool_representations(
    model,
    dataloader,
    source_df,
    domain_name,
    experiment_name,
    seed
):

    model.eval()


    captured_features = []


    def avgpool_hook(
        module,
        input_data,
        output
    ):

        captured_features.append(
            output
            .detach()
            .cpu()
        )


    hook_handle = (
        model.avgpool
        .register_forward_hook(
            avgpool_hook
        )
    )


    metadata_records = []


    with torch.no_grad():

        for images, labels, indices in dataloader:

            images = images.to(
                DEVICE
            )


            captured_features.clear()


            _ = model(
                images
            )


            batch_features = (

                captured_features[
                    0
                ]

                .flatten(
                    start_dim=1
                )

                .numpy()
            )


            labels_np = (
                labels
                .numpy()
            )


            indices_np = (
                indices
                .numpy()
            )


            for batch_position, source_index in enumerate(
                indices_np
            ):

                row = source_df.iloc[
                    int(
                        source_index
                    )
                ]


                metadata_records.append({

                    "Experiment":
                        experiment_name,

                    "Seed":
                        int(
                            seed
                        ),

                    "Domain":
                        domain_name,

                    "Source_Index":
                        int(
                            source_index
                        ),

                    "image_path":
                        row[
                            "image_path"
                        ],

                    "md5hash":
                        row[
                            "md5hash"
                        ],

                    "skin_tone":
                        row[
                            "skin_tone"
                        ],

                    "True_Label":
                        int(
                            labels_np[
                                batch_position
                            ]
                        )
                })


            if (
                "all_features"
                not in locals()
            ):

                all_features = [
                    batch_features
                ]

            else:

                all_features.append(
                    batch_features
                )


    hook_handle.remove()


    features_array = np.concatenate(
        all_features,
        axis=0
    )


    metadata_df = pd.DataFrame(
        metadata_records
    )


    return (
        features_array,
        metadata_df
    )

In [ ]:
# ============================================
# Section 9.4
# Final Robust Avgpool Extraction Function
# ============================================

def extract_avgpool_representations(
    model,
    dataloader,
    source_df,
    domain_name,
    experiment_name,
    seed
):

    model.eval()


    feature_batches = []

    metadata_records = []


    captured_output = {}


    def avgpool_hook(
        module,
        input_data,
        output
    ):

        captured_output[
            "features"
        ] = (
            output
            .detach()
            .cpu()
        )


    hook_handle = (
        model.avgpool
        .register_forward_hook(
            avgpool_hook
        )
    )


    try:

        with torch.no_grad():

            for images, labels, indices in dataloader:

                images = images.to(
                    DEVICE
                )


                captured_output.clear()


                _ = model(
                    images
                )


                if (
                    "features"
                    not in captured_output
                ):

                    raise RuntimeError(
                        "Avgpool hook did not capture features."
                    )


                batch_features = (

                    captured_output[
                        "features"
                    ]

                    .flatten(
                        start_dim=1
                    )

                    .numpy()
                )


                feature_batches.append(
                    batch_features
                )


                labels_np = (
                    labels
                    .numpy()
                )


                indices_np = (
                    indices
                    .numpy()
                )


                for batch_position, source_index in enumerate(
                    indices_np
                ):

                    row = source_df.iloc[
                        int(
                            source_index
                        )
                    ]


                    metadata_records.append({

                        "Experiment":
                            experiment_name,

                        "Seed":
                            int(
                                seed
                            ),

                        "Domain":
                            domain_name,

                        "Source_Index":
                            int(
                                source_index
                            ),

                        "image_path":
                            row[
                                "image_path"
                            ],

                        "md5hash":
                            row[
                                "md5hash"
                            ],

                        "skin_tone":
                            row[
                                "skin_tone"
                            ],

                        "True_Label":
                            int(
                                labels_np[
                                    batch_position
                                ]
                            )
                    })


    finally:

        hook_handle.remove()


    features_array = np.concatenate(
        feature_batches,
        axis=0
    )


    metadata_df = pd.DataFrame(
        metadata_records
    )


    return (
        features_array,
        metadata_df
    )

In [ ]:
# ============================================
# Section 9.5
# Define Representation Artifact Paths
# ============================================

def representation_paths(
    experiment_name,
    seed,
    domain_name
):

    prefix = (
        f"{experiment_name}"
        f"_seed_{seed}"
        f"_{domain_name.lower()}"
    )


    feature_file = (

        PHASE5_REPRESENTATION_PATH

        / f"{prefix}_avgpool.npy"
    )


    metadata_file = (

        PHASE5_REPRESENTATION_PATH

        / f"{prefix}_metadata.csv"
    )


    return (
        feature_file,
        metadata_file
    )


def representation_complete(
    experiment_name,
    seed,
    domain_name,
    expected_rows
):

    feature_file, metadata_file = (
        representation_paths(
            experiment_name,
            seed,
            domain_name
        )
    )


    if (
        not feature_file.exists()
        or
        not metadata_file.exists()
    ):

        return False


    try:

        features = np.load(
            feature_file,
            mmap_mode="r"
        )

        metadata = pd.read_csv(
            metadata_file
        )

    except Exception:

        return False


    return (

        features.shape
        ==
        (
            expected_rows,
            2048
        )

        and

        len(
            metadata
        )
        ==
        expected_rows
    )

In [ ]:
# ============================================
# Section 9.6
# Avgpool Extraction Sanity Check
# One Model / Fair Domain
# ============================================

TEST_EXPERIMENT = (
    "M1_75F_25D"
)

TEST_SEED = 42


test_checkpoint = (

    CORRECTED_PHASE4_MODEL_PATH

    / f"{TEST_EXPERIMENT}_seed_{TEST_SEED}.pth"
)


test_model = (
    create_phase4_resnet50()
)


test_state = torch.load(

    test_checkpoint,

    map_location=DEVICE
)


test_model.load_state_dict(
    test_state
)


test_features, test_metadata = (
    extract_avgpool_representations(

        model=test_model,

        dataloader=fair_test_loader,

        source_df=fair_test_df,

        domain_name="Fair",

        experiment_name=TEST_EXPERIMENT,

        seed=TEST_SEED
    )
)


print(
    "Feature shape:",
    test_features.shape
)

print(
    "Metadata shape:",
    test_metadata.shape
)

print(
    "Feature dtype:",
    test_features.dtype
)

print(
    "Finite values:",
    np.isfinite(
        test_features
    ).all()
)


print(
    "Feature minimum:",
    test_features.min()
)

print(
    "Feature maximum:",
    test_features.max()
)


del (
    test_model,
    test_state
)

if torch.cuda.is_available():

    torch.cuda.empty_cache()

In [ ]:
# ============================================
# Section 9.7
# Extract Avgpool Representations
# All 15 Corrected Models
# Resume-Safe
# ============================================

representation_run_rows = []


for experiment_name in EXPERIMENTS:

    for seed in SEEDS:

        print(
            "\n" + "=" * 90
        )

        print(
            f"{experiment_name} | Seed {seed}"
        )

        print(
            "=" * 90
        )


        fair_complete = (
            representation_complete(

                experiment_name,

                seed,

                "Fair",

                len(
                    fair_test_df
                )
            )
        )


        dark_complete = (
            representation_complete(

                experiment_name,

                seed,

                "Dark",

                len(
                    dark_test_df
                )
            )
        )


        if (
            fair_complete
            and
            dark_complete
        ):

            print(
                "✓ SKIP — both domains already complete."
            )


            representation_run_rows.append({

                "Experiment":
                    experiment_name,

                "Seed":
                    seed,

                "Status":
                    "Existing"
            })


            continue


        checkpoint_file = (

            CORRECTED_PHASE4_MODEL_PATH

            / (
                f"{experiment_name}"
                f"_seed_{seed}.pth"
            )
        )


        model = (
            create_phase4_resnet50()
        )


        state_dict = torch.load(

            checkpoint_file,

            map_location=DEVICE
        )


        model.load_state_dict(
            state_dict
        )


        model.eval()


        # ------------------------------------
        # FAIR
        # ------------------------------------

        if not fair_complete:

            fair_features, fair_metadata = (
                extract_avgpool_representations(

                    model=model,

                    dataloader=fair_test_loader,

                    source_df=fair_test_df,

                    domain_name="Fair",

                    experiment_name=experiment_name,

                    seed=seed
                )
            )


            fair_feature_file, fair_metadata_file = (
                representation_paths(

                    experiment_name,

                    seed,

                    "Fair"
                )
            )


            np.save(
                fair_feature_file,
                fair_features
            )


            fair_metadata.to_csv(

                fair_metadata_file,

                index=False
            )


            print(
                "✓ Fair:",
                fair_features.shape
            )


            del (
                fair_features,
                fair_metadata
            )


        else:

            print(
                "✓ Fair already exists."
            )


        # ------------------------------------
        # DARK
        # ------------------------------------

        if not dark_complete:

            dark_features, dark_metadata = (
                extract_avgpool_representations(

                    model=model,

                    dataloader=dark_test_loader,

                    source_df=dark_test_df,

                    domain_name="Dark",

                    experiment_name=experiment_name,

                    seed=seed
                )
            )


            dark_feature_file, dark_metadata_file = (
                representation_paths(

                    experiment_name,

                    seed,

                    "Dark"
                )
            )


            np.save(
                dark_feature_file,
                dark_features
            )


            dark_metadata.to_csv(

                dark_metadata_file,

                index=False
            )


            print(
                "✓ Dark:",
                dark_features.shape
            )


            del (
                dark_features,
                dark_metadata
            )


        else:

            print(
                "✓ Dark already exists."
            )


        representation_run_rows.append({

            "Experiment":
                experiment_name,

            "Seed":
                seed,

            "Status":
                "Generated"
        })


        del (
            model,
            state_dict
        )


        if torch.cuda.is_available():

            torch.cuda.empty_cache()


representation_run_summary_df = (
    pd.DataFrame(
        representation_run_rows
    )
)


display(
    representation_run_summary_df
)

In [ ]:
# ============================================
# Section 9.8
# Audit All Representation Artifacts
# ============================================

representation_audit_rows = []


for experiment_name in EXPERIMENTS:

    for seed in SEEDS:

        for domain_name, expected_rows in [

            (
                "Fair",
                len(
                    fair_test_df
                )
            ),

            (
                "Dark",
                len(
                    dark_test_df
                )
            )
        ]:

            feature_file, metadata_file = (
                representation_paths(

                    experiment_name,

                    seed,

                    domain_name
                )
            )


            exists = (

                feature_file.exists()
                and
                metadata_file.exists()
            )


            if exists:

                features = np.load(

                    feature_file,

                    mmap_mode="r"
                )


                metadata = pd.read_csv(
                    metadata_file
                )


                shape_correct = (

                    features.shape
                    ==
                    (
                        expected_rows,
                        2048
                    )
                )


                metadata_correct = (

                    len(
                        metadata
                    )
                    ==
                    expected_rows
                )


                finite = bool(

                    np.isfinite(
                        features
                    )
                    .all()
                )


            else:

                shape_correct = False
                metadata_correct = False
                finite = False


            representation_audit_rows.append({

                "Experiment":
                    experiment_name,

                "Seed":
                    seed,

                "Domain":
                    domain_name,

                "Expected_Rows":
                    expected_rows,

                "Exists":
                    exists,

                "Shape_Correct":
                    shape_correct,

                "Metadata_Correct":
                    metadata_correct,

                "Finite":
                    finite
            })


representation_audit_df = (
    pd.DataFrame(
        representation_audit_rows
    )
)


display(
    representation_audit_df
)


print(
    "\nComplete artifacts:",
    int(

        (
            representation_audit_df[
                "Exists"
            ]

            &

            representation_audit_df[
                "Shape_Correct"
            ]

            &

            representation_audit_df[
                "Metadata_Correct"
            ]

            &

            representation_audit_df[
                "Finite"
            ]
        )
        .sum()
    ),

    "/ 30"
)

In [ ]:
# ============================================
# Section 9.9
# Create Representation Manifest
# ============================================

representation_manifest_rows = []


for experiment_name in EXPERIMENTS:

    for seed in SEEDS:

        for domain_name in [
            "Fair",
            "Dark"
        ]:

            feature_file, metadata_file = (
                representation_paths(

                    experiment_name,

                    seed,

                    domain_name
                )
            )


            features = np.load(
                feature_file,
                mmap_mode="r"
            )


            metadata = pd.read_csv(
                metadata_file
            )


            representation_manifest_rows.append({

                "Experiment":
                    experiment_name,

                "Seed":
                    seed,

                "Domain":
                    domain_name,

                "Rows":
                    features.shape[0],

                "Feature_Dimension":
                    features.shape[1],

                "Feature_File":
                    str(
                        feature_file
                    ),

                "Metadata_File":
                    str(
                        metadata_file
                    ),

                "Benign":
                    int(
                        metadata[
                            "True_Label"
                        ]
                        .eq(0)
                        .sum()
                    ),

                "Malignant":
                    int(
                        metadata[
                            "True_Label"
                        ]
                        .eq(1)
                        .sum()
                    )
            })


representation_manifest_df = (
    pd.DataFrame(
        representation_manifest_rows
    )
)


display(
    representation_manifest_df
)


representation_manifest_df.to_csv(

    PHASE5_TABLE_PATH
    / "Phase5_Corrected_Representation_Manifest.csv",

    index=False
)


representation_audit_df.to_csv(

    PHASE5_TABLE_PATH
    / "Phase5_Corrected_Representation_Audit.csv",

    index=False
)


print(
    "✓ Representation manifest and audit saved."
)


# Section 10: Class-Conditional Learned Representation Shift

The corrected Phase 5 evaluates Fair–Dark discrepancy in the final ResNet50 avgpool representation:

$$
z\in\mathbb{R}^{2048}.
$$

Two representation-shift measures are considered:

1. centroid cosine distance — descriptive,
2. Gaussian-kernel unbiased Maximum Mean Discrepancy squared (MMD²) — primary inferential measure.

Because Fair and Dark test sets differ in disease composition, representation discrepancy is evaluated separately for:

$$
Y=0
$$

and:

$$
Y=1.
$$

The Gaussian kernel is:

$$
k(x,y)
=
\exp
\left(
-\frac{\|x-y\|^2}{2\sigma^2}
\right).
$$

Primary inference uses a common bandwidth separately for benign and malignant representations.

The common bandwidth is estimated from the corrected models before inspecting corrected MMD outcomes.

This ensures that MMD values within a disease class are evaluated under the same kernel scale across all 15 trained models.

In [ ]:
# ============================================
# Section 10.1
# Helper to Load Fair and Dark Representations
# ============================================

def load_representation_pair(
    experiment_name,
    seed
):

    fair_feature_file, fair_metadata_file = (
        representation_paths(
            experiment_name,
            seed,
            "Fair"
        )
    )

    dark_feature_file, dark_metadata_file = (
        representation_paths(
            experiment_name,
            seed,
            "Dark"
        )
    )


    fair_features = np.load(
        fair_feature_file
    )

    dark_features = np.load(
        dark_feature_file
    )


    fair_metadata = pd.read_csv(
        fair_metadata_file
    )

    dark_metadata = pd.read_csv(
        dark_metadata_file
    )


    return (
        fair_features,
        fair_metadata,
        dark_features,
        dark_metadata
    )

In [ ]:
# ============================================
# Section 10.2
# Pairwise Distance and Kernel Utilities
# ============================================

def squared_euclidean_matrix(
    x,
    y
):

    x = np.asarray(
        x,
        dtype=np.float64
    )

    y = np.asarray(
        y,
        dtype=np.float64
    )


    x_norm = np.sum(
        x ** 2,
        axis=1,
        keepdims=True
    )

    y_norm = np.sum(
        y ** 2,
        axis=1,
        keepdims=True
    ).T


    distances = (
        x_norm
        +
        y_norm
        -
        2.0
        *
        np.matmul(
            x,
            y.T
        )
    )


    return np.maximum(
        distances,
        0.0
    )


def gaussian_kernel_from_sqdist(
    squared_distances,
    sigma
):

    return np.exp(

        -squared_distances

        /

        (
            2.0
            *
            sigma ** 2
        )
    )

In [ ]:
# ============================================
# Section 10.3
# Median-Heuristic Bandwidth Function
# ============================================

def median_heuristic_bandwidth(
    features
):

    features = np.asarray(
        features,
        dtype=np.float64
    )


    squared_distances = (
        squared_euclidean_matrix(
            features,
            features
        )
    )


    upper_triangle = (

        squared_distances[

            np.triu_indices(
                len(features),
                k=1
            )
        ]
    )


    positive_distances = (
        upper_triangle[
            upper_triangle
            >
            0
        ]
    )


    if len(
        positive_distances
    ) == 0:

        raise ValueError(
            "No positive pairwise distances."
        )


    median_squared_distance = np.median(
        positive_distances
    )


    sigma = np.sqrt(
        median_squared_distance
    )


    return float(
        sigma
    )

In [ ]:
# ============================================
# Section 10.4
# Estimate Model-Specific Class Bandwidths
# ============================================

bandwidth_rows = []


for experiment_name in EXPERIMENT_ORDER:

    for seed in SEEDS:

        (
            fair_features,
            fair_metadata,
            dark_features,
            dark_metadata

        ) = load_representation_pair(

            experiment_name,
            seed
        )


        for class_label, target_value in [

            ("Benign", 0),
            ("Malignant", 1)

        ]:

            fair_class_features = (

                fair_features[

                    fair_metadata[
                        "True_Label"
                    ]
                    .values
                    ==
                    target_value
                ]
            )


            dark_class_features = (

                dark_features[

                    dark_metadata[
                        "True_Label"
                    ]
                    .values
                    ==
                    target_value
                ]
            )


            combined_features = np.concatenate(

                [
                    fair_class_features,
                    dark_class_features
                ],

                axis=0
            )


            sigma = median_heuristic_bandwidth(
                combined_features
            )


            bandwidth_rows.append({

                "Experiment":
                    experiment_name,

                "Seed":
                    seed,

                "Class":
                    class_label,

                "Fair_N":
                    len(
                        fair_class_features
                    ),

                "Dark_N":
                    len(
                        dark_class_features
                    ),

                "Model_Specific_Sigma":
                    sigma
            })


model_specific_bandwidth_df = (
    pd.DataFrame(
        bandwidth_rows
    )
)


display(
    model_specific_bandwidth_df
)


print(
    "\nBandwidth range:"
)

print(
    model_specific_bandwidth_df[
        "Model_Specific_Sigma"
    ]
    .min(),
    "to",
    model_specific_bandwidth_df[
        "Model_Specific_Sigma"
    ]
    .max()
)

In [ ]:
# ============================================
# Section 10.5
# Define Common Class-Specific Bandwidths
# ============================================

common_bandwidth_rows = []


COMMON_BANDWIDTHS = {}


for class_label in [
    "Benign",
    "Malignant"
]:

    class_bandwidths = (

        model_specific_bandwidth_df.loc[

            model_specific_bandwidth_df[
                "Class"
            ]
            ==
            class_label,

            "Model_Specific_Sigma"
        ]

        .astype(float)

        .values
    )


    common_sigma = float(
        np.median(
            class_bandwidths
        )
    )


    COMMON_BANDWIDTHS[
        class_label
    ] = common_sigma


    common_bandwidth_rows.append({

        "Class":
            class_label,

        "Common_Sigma":
            common_sigma,

        "Model_Specific_Min":
            float(
                np.min(
                    class_bandwidths
                )
            ),

        "Model_Specific_Max":
            float(
                np.max(
                    class_bandwidths
                )
            ),

        "Model_Specific_Mean":
            float(
                np.mean(
                    class_bandwidths
                )
            ),

        "Model_Specific_Median":
            float(
                np.median(
                    class_bandwidths
                )
            )
    })


common_bandwidth_df = (
    pd.DataFrame(
        common_bandwidth_rows
    )
)


display(
    common_bandwidth_df
)


common_bandwidth_df.to_csv(

    PHASE5_TABLE_PATH
    / "Phase5_Corrected_Common_MMD_Bandwidths.csv",

    index=False
)


model_specific_bandwidth_df.to_csv(

    PHASE5_TABLE_PATH
    / "Phase5_Corrected_Model_Specific_MMD_Bandwidths.csv",

    index=False
)


print(
    "\nLocked corrected bandwidths:"
)

print(
    COMMON_BANDWIDTHS
)

In [ ]:
# ============================================
# Section 10.6
# Unbiased Gaussian MMD Squared
# ============================================

def unbiased_mmd_squared(
    x,
    y,
    sigma
):

    x = np.asarray(
        x,
        dtype=np.float64
    )

    y = np.asarray(
        y,
        dtype=np.float64
    )


    m = len(x)
    n = len(y)


    if m < 2 or n < 2:

        raise ValueError(
            "MMD requires at least 2 samples per group."
        )


    d_xx = squared_euclidean_matrix(
        x,
        x
    )

    d_yy = squared_euclidean_matrix(
        y,
        y
    )

    d_xy = squared_euclidean_matrix(
        x,
        y
    )


    k_xx = gaussian_kernel_from_sqdist(
        d_xx,
        sigma
    )

    k_yy = gaussian_kernel_from_sqdist(
        d_yy,
        sigma
    )

    k_xy = gaussian_kernel_from_sqdist(
        d_xy,
        sigma
    )


    term_xx = (

        (
            k_xx.sum()
            -
            np.trace(
                k_xx
            )
        )

        /

        (
            m
            *
            (m - 1)
        )
    )


    term_yy = (

        (
            k_yy.sum()
            -
            np.trace(
                k_yy
            )
        )

        /

        (
            n
            *
            (n - 1)
        )
    )


    term_xy = (
        k_xy.mean()
    )


    mmd_squared = (

        term_xx
        +
        term_yy
        -
        2.0
        *
        term_xy
    )


    return float(
        mmd_squared
    )

In [ ]:
# ============================================
# Section 10.6
# Unbiased Gaussian MMD Squared
# ============================================

def unbiased_mmd_squared(
    x,
    y,
    sigma
):

    x = np.asarray(
        x,
        dtype=np.float64
    )

    y = np.asarray(
        y,
        dtype=np.float64
    )


    m = len(x)
    n = len(y)


    if m < 2 or n < 2:

        raise ValueError(
            "MMD requires at least 2 samples per group."
        )


    d_xx = squared_euclidean_matrix(
        x,
        x
    )

    d_yy = squared_euclidean_matrix(
        y,
        y
    )

    d_xy = squared_euclidean_matrix(
        x,
        y
    )


    k_xx = gaussian_kernel_from_sqdist(
        d_xx,
        sigma
    )

    k_yy = gaussian_kernel_from_sqdist(
        d_yy,
        sigma
    )

    k_xy = gaussian_kernel_from_sqdist(
        d_xy,
        sigma
    )


    term_xx = (

        (
            k_xx.sum()
            -
            np.trace(
                k_xx
            )
        )

        /

        (
            m
            *
            (m - 1)
        )
    )


    term_yy = (

        (
            k_yy.sum()
            -
            np.trace(
                k_yy
            )
        )

        /

        (
            n
            *
            (n - 1)
        )
    )


    term_xy = (
        k_xy.mean()
    )


    mmd_squared = (

        term_xx
        +
        term_yy
        -
        2.0
        *
        term_xy
    )


    return float(
        mmd_squared
    )

In [ ]:
# ============================================
# Section 10.7
# Centroid Cosine Distance
# Descriptive Measure
# ============================================

def centroid_cosine_distance(
    x,
    y
):

    centroid_x = np.mean(
        x,
        axis=0
    )

    centroid_y = np.mean(
        y,
        axis=0
    )


    denominator = (

        np.linalg.norm(
            centroid_x
        )

        *

        np.linalg.norm(
            centroid_y
        )
    )


    if denominator == 0:

        return np.nan


    cosine_similarity = (

        np.dot(
            centroid_x,
            centroid_y
        )

        /

        denominator
    )


    return float(
        1.0
        -
        cosine_similarity
    )

In [ ]:
# ============================================
# Section 10.8
# Compute Class-Conditional Representation Shift
# ============================================

representation_shift_rows = []


for experiment_name in EXPERIMENT_ORDER:

    for seed in SEEDS:

        (
            fair_features,
            fair_metadata,
            dark_features,
            dark_metadata

        ) = load_representation_pair(

            experiment_name,
            seed
        )


        for class_label, target_value in [

            ("Benign", 0),
            ("Malignant", 1)

        ]:

            fair_class = (

                fair_features[

                    fair_metadata[
                        "True_Label"
                    ]
                    .values
                    ==
                    target_value
                ]
            )


            dark_class = (

                dark_features[

                    dark_metadata[
                        "True_Label"
                    ]
                    .values
                    ==
                    target_value
                ]
            )


            sigma = (
                COMMON_BANDWIDTHS[
                    class_label
                ]
            )


            common_mmd_squared = (
                unbiased_mmd_squared(

                    fair_class,
                    dark_class,
                    sigma
                )
            )


            model_specific_sigma = float(

                model_specific_bandwidth_df.loc[

                    (
                        model_specific_bandwidth_df[
                            "Experiment"
                        ]
                        ==
                        experiment_name
                    )

                    &

                    (
                        model_specific_bandwidth_df[
                            "Seed"
                        ]
                        ==
                        seed
                    )

                    &

                    (
                        model_specific_bandwidth_df[
                            "Class"
                        ]
                        ==
                        class_label
                    ),

                    "Model_Specific_Sigma"
                ]
                .iloc[0]
            )


            adaptive_mmd_squared = (
                unbiased_mmd_squared(

                    fair_class,
                    dark_class,
                    model_specific_sigma
                )
            )


            cosine_distance = (
                centroid_cosine_distance(

                    fair_class,
                    dark_class
                )
            )


            representation_shift_rows.append({

                "Experiment":
                    experiment_name,

                "Seed":
                    seed,

                "Class":
                    class_label,

                "Fair_N":
                    len(
                        fair_class
                    ),

                "Dark_N":
                    len(
                        dark_class
                    ),

                "Common_Sigma":
                    sigma,

                "Common_MMD2":
                    common_mmd_squared,

                "Model_Specific_Sigma":
                    model_specific_sigma,

                "Adaptive_MMD2":
                    adaptive_mmd_squared,

                "Centroid_Cosine_Distance":
                    cosine_distance
            })


representation_shift_df = (
    pd.DataFrame(
        representation_shift_rows
    )
)


display(
    representation_shift_df
)


print(
    "\nShape:",
    representation_shift_df.shape
)

In [ ]:
# ============================================
# Section 10.9
# Summarize Common-Bandwidth MMD Squared
# ============================================

common_mmd_summary_df = (

    representation_shift_df

    .groupby(
        [
            "Experiment",
            "Class"
        ]
    )

    .agg(

        Mean_MMD2=(
            "Common_MMD2",
            "mean"
        ),

        SD_MMD2=(
            "Common_MMD2",
            "std"
        ),

        Mean_Cosine_Distance=(
            "Centroid_Cosine_Distance",
            "mean"
        ),

        SD_Cosine_Distance=(
            "Centroid_Cosine_Distance",
            "std"
        )
    )

    .reset_index()
)


display(
    common_mmd_summary_df
)

In [ ]:
# ============================================
# Section 10.10
# Exact Friedman Tests
# Common-Bandwidth Class-Conditional MMD
# ============================================

mmd_stat_rows = []


for class_label in [
    "Benign",
    "Malignant"
]:

    class_df = (

        representation_shift_df[

            representation_shift_df[
                "Class"
            ]
            ==
            class_label
        ]
    )


    matrix_rows = []


    for seed in SEEDS:

        seed_values = []


        for experiment_name in (
            EXPERIMENT_ORDER
        ):

            value = float(

                class_df.loc[

                    (
                        class_df[
                            "Experiment"
                        ]
                        ==
                        experiment_name
                    )

                    &

                    (
                        class_df[
                            "Seed"
                        ]
                        ==
                        seed
                    ),

                    "Common_MMD2"
                ]
                .iloc[0]
            )


            seed_values.append(
                value
            )


        matrix_rows.append(
            seed_values
        )


    matrix = np.asarray(
        matrix_rows,
        dtype=float
    )


    result = exact_friedman_test(
        matrix
    )


    mmd_stat_rows.append({

        "Outcome":
            "Common_Bandwidth_MMD2",

        "Class":
            class_label,

        **result
    })


common_mmd_friedman_df = (
    pd.DataFrame(
        mmd_stat_rows
    )
)


display(
    common_mmd_friedman_df
)

In [ ]:
# ============================================
# Section 10.11
# Holm Correction Across MMD Hypotheses
# ============================================

reject, p_holm, _, _ = multipletests(

    common_mmd_friedman_df[
        "Exact_P"
    ]
    .values,

    alpha=0.05,

    method="holm"
)


common_mmd_friedman_df[
    "Holm_P"
] = p_holm


common_mmd_friedman_df[
    "Supported_After_Holm"
] = reject


display(
    common_mmd_friedman_df
)

In [ ]:
# ============================================
# Section 10.12
# Save Corrected Representation-Shift Results
# ============================================

representation_shift_df.to_csv(

    PHASE5_TABLE_PATH
    / "Phase5_Corrected_Class_Conditional_Representation_Shift.csv",

    index=False
)


common_mmd_summary_df.to_csv(

    PHASE5_TABLE_PATH
    / "Phase5_Corrected_Common_Bandwidth_MMD_Summary.csv",

    index=False
)


common_mmd_friedman_df.to_csv(

    PHASE5_TABLE_PATH
    / "Phase5_Corrected_Common_Bandwidth_MMD_Exact_Friedman.csv",

    index=False
)


print(
    "✓ Corrected representation-shift artifacts saved."
)

## Section 10.13: Corrected Representation-Shift Result

Class-conditional Fair–Dark avgpool representation discrepancy was evaluated using unbiased Gaussian-kernel MMD² with common class-specific bandwidths.

For benign lesions:

$$
Q=5.2
$$

$$
p_{\mathrm{exact}}=0.093364
$$

$$
W=0.52
$$

and:

$$
p_{\mathrm{Holm}}=0.186728.
$$

For malignant lesions:

$$
Q=2.8
$$

$$
p_{\mathrm{exact}}=0.367284
$$

$$
W=0.28
$$

and:

$$
p_{\mathrm{Holm}}=0.367284.
$$

Therefore, neither class-conditional representation-shift outcome remained statistically supported after multiplicity correction.

Descriptively, benign MMD² increased across M1, M2, and M3, whereas malignant MMD² decreased. These patterns are retained as descriptive observations only.

Because neither omnibus hypothesis was statistically supported, confirmatory pairwise MMD post-hoc tests were not pursued.

# Section 11: PCA Visualization of Learned Representations

Principal Component Analysis is used to visualize the corrected avgpool feature distributions.

PCA is fitted separately within each trained model.

Fair and Dark representations from the same model are concatenated before fitting PCA so that the two domains are visualized in a shared within-model coordinate system.

Raw representations from different independently trained models are not combined into one global PCA space.

For each model:

$$
Z=
\begin{bmatrix}
Z_F\\
Z_D
\end{bmatrix}
$$

is projected onto:

$$
PC_1,\ PC_2.
$$

PCA is treated strictly as a descriptive visualization.

Variance explained by the first two components is recorded, but the PCA plots are not used as inferential evidence of domain separation.

In [ ]:
# ============================================
# Section 11.1
# Define PCA Artifact Paths
# ============================================

from sklearn.decomposition import PCA
import matplotlib.pyplot as plt


PHASE5_PCA_PATH = (
    CORRECTED_PHASE5_PATH
    / "representation_pca"
)

PHASE5_PCA_PATH.mkdir(
    parents=True,
    exist_ok=True
)


def pca_coordinate_path(
    experiment_name,
    seed
):

    return (

        PHASE5_PCA_PATH

        / (
            f"{experiment_name}"
            f"_seed_{seed}"
            f"_pca_coordinates.csv"
        )
    )


def pca_figure_path(
    experiment_name,
    seed
):

    return (

        PHASE5_PCA_PATH

        / (
            f"{experiment_name}"
            f"_seed_{seed}"
            f"_pca.png"
        )
    )

In [ ]:
# ============================================
# Section 11.2
# Compute Within-Model PCA
# ============================================

pca_manifest_rows = []


for experiment_name in EXPERIMENT_ORDER:

    for seed in SEEDS:

        (
            fair_features,
            fair_metadata,
            dark_features,
            dark_metadata

        ) = load_representation_pair(

            experiment_name,
            seed
        )


        combined_features = np.concatenate(

            [
                fair_features,
                dark_features
            ],

            axis=0
        )


        combined_domain = np.array(

            ["Fair"] * len(
                fair_features
            )

            +

            ["Dark"] * len(
                dark_features
            )
        )


        combined_labels = np.concatenate(

            [
                fair_metadata[
                    "True_Label"
                ].values,

                dark_metadata[
                    "True_Label"
                ].values
            ]
        )


        combined_md5 = np.concatenate(

            [
                fair_metadata[
                    "md5hash"
                ].values,

                dark_metadata[
                    "md5hash"
                ].values
            ]
        )


        combined_paths = np.concatenate(

            [
                fair_metadata[
                    "image_path"
                ].values,

                dark_metadata[
                    "image_path"
                ].values
            ]
        )


        pca = PCA(
            n_components=2
        )


        coordinates = (
            pca.fit_transform(
                combined_features
            )
        )


        coordinate_df = pd.DataFrame({

            "Experiment":
                experiment_name,

            "Seed":
                seed,

            "Domain":
                combined_domain,

            "True_Label":
                combined_labels,

            "md5hash":
                combined_md5,

            "image_path":
                combined_paths,

            "PC1":
                coordinates[:, 0],

            "PC2":
                coordinates[:, 1]
        })


        coordinate_file = (
            pca_coordinate_path(
                experiment_name,
                seed
            )
        )


        coordinate_df.to_csv(

            coordinate_file,

            index=False
        )


        explained_variance = (
            pca.explained_variance_ratio_
        )


        pca_manifest_rows.append({

            "Experiment":
                experiment_name,

            "Seed":
                seed,

            "PC1_Explained_Variance":
                float(
                    explained_variance[0]
                ),

            "PC2_Explained_Variance":
                float(
                    explained_variance[1]
                ),

            "PC1_PC2_Total":
                float(
                    explained_variance.sum()
                ),

            "Coordinate_File":
                str(
                    coordinate_file
                )
        })


pca_manifest_df = (
    pd.DataFrame(
        pca_manifest_rows
    )
)


display(
    pca_manifest_df
)

In [ ]:
# ============================================
# Section 11.3
# Generate PCA Figures
# ============================================

pca_figure_rows = []


for experiment_name in EXPERIMENT_ORDER:

    for seed in SEEDS:

        coordinate_file = (
            pca_coordinate_path(
                experiment_name,
                seed
            )
        )


        coordinate_df = pd.read_csv(
            coordinate_file
        )


        figure_file = (
            pca_figure_path(
                experiment_name,
                seed
            )
        )


        plt.figure(
            figsize=(7, 6)
        )


        for domain_name in [
            "Fair",
            "Dark"
        ]:

            subset = (

                coordinate_df[

                    coordinate_df[
                        "Domain"
                    ]
                    ==
                    domain_name
                ]
            )


            plt.scatter(

                subset[
                    "PC1"
                ],

                subset[
                    "PC2"
                ],

                alpha=0.55,

                s=20,

                label=domain_name
            )


        plt.xlabel(
            "Principal Component 1"
        )

        plt.ylabel(
            "Principal Component 2"
        )

        plt.title(
            f"{experiment_name} | Seed {seed}"
        )

        plt.legend()

        plt.tight_layout()


        plt.savefig(

            figure_file,

            dpi=300,

            bbox_inches="tight"
        )


        plt.close()


        pca_figure_rows.append({

            "Experiment":
                experiment_name,

            "Seed":
                seed,

            "Figure_File":
                str(
                    figure_file
                )
        })


pca_figure_manifest_df = (
    pd.DataFrame(
        pca_figure_rows
    )
)


print(
    "PCA figures generated:",
    len(
        pca_figure_manifest_df
    )
)

In [ ]:
# ============================================
# Section 11.4
# Summarize PCA Explained Variance
# ============================================

pca_variance_summary_df = (

    pca_manifest_df

    .groupby(
        "Experiment"
    )

    .agg(

        PC1_Mean=(
            "PC1_Explained_Variance",
            "mean"
        ),

        PC1_SD=(
            "PC1_Explained_Variance",
            "std"
        ),

        PC2_Mean=(
            "PC2_Explained_Variance",
            "mean"
        ),

        PC2_SD=(
            "PC2_Explained_Variance",
            "std"
        ),

        Total_PC1_PC2_Mean=(
            "PC1_PC2_Total",
            "mean"
        ),

        Total_PC1_PC2_SD=(
            "PC1_PC2_Total",
            "std"
        )
    )

    .reset_index()
)


display(
    pca_variance_summary_df
)

In [ ]:
# ============================================
# Section 11.5
# Save PCA Manifests
# ============================================

pca_manifest_df.to_csv(

    PHASE5_TABLE_PATH
    / "Phase5_Corrected_PCA_Manifest.csv",

    index=False
)


pca_variance_summary_df.to_csv(

    PHASE5_TABLE_PATH
    / "Phase5_Corrected_PCA_Variance_Summary.csv",

    index=False
)


pca_figure_manifest_df.to_csv(

    PHASE5_TABLE_PATH
    / "Phase5_Corrected_PCA_Figure_Manifest.csv",

    index=False
)


print(
    "✓ PCA artifacts saved."
)

# Section 12: Final Statistical Synthesis

Corrected Phase 5 evaluated how controlled Fair/Dark training composition influenced:

1. prediction behavior,
2. predictive uncertainty,
3. probabilistic prediction error,
4. malignant false-negative persistence,
5. learned avgpool representations.

The independent replication unit throughout inferential analyses was the seed-specific trained model:

$$
n=5
$$

per training composition.

Exact within-seed permutation Friedman tests were used for predefined omnibus hypotheses, with Holm correction within each prespecified hypothesis family.

Image-level analyses were treated descriptively and were not interpreted as thousands of independent experimental replicates.

In [ ]:
# ============================================
# Section 12.1
# Build Final Corrected Phase 5
# Statistical Summary
# ============================================

final_statistical_rows = []


# --------------------------------------------
# Prediction behavior
# --------------------------------------------

for _, row in (
    prediction_behavior_friedman_df
    .iterrows()
):

    final_statistical_rows.append({

        "Analysis_Family":
            "Prediction Behavior",

        "Outcome":
            row[
                "Outcome"
            ],

        "Class":
            row[
                "Class"
            ],

        "Friedman_Q":
            row[
                "Friedman_Q"
            ],

        "Exact_P":
            row[
                "Exact_P"
            ],

        "Kendalls_W":
            row[
                "Kendalls_W"
            ],

        "Holm_P":
            row[
                "Holm_P"
            ],

        "Supported_After_Holm":
            row[
                "Supported_After_Holm"
            ]
    })


# --------------------------------------------
# Brier
# --------------------------------------------

for _, row in (
    class_brier_friedman_df
    .iterrows()
):

    final_statistical_rows.append({

        "Analysis_Family":
            "Class-Conditional Brier",

        "Outcome":
            row[
                "Outcome"
            ],

        "Class":
            row[
                "Class"
            ],

        "Friedman_Q":
            row[
                "Friedman_Q"
            ],

        "Exact_P":
            row[
                "Exact_P"
            ],

        "Kendalls_W":
            row[
                "Kendalls_W"
            ],

        "Holm_P":
            row[
                "Holm_P"
            ],

        "Supported_After_Holm":
            row[
                "Supported_After_Holm"
            ]
    })


# --------------------------------------------
# Representation MMD
# --------------------------------------------

for _, row in (
    common_mmd_friedman_df
    .iterrows()
):

    final_statistical_rows.append({

        "Analysis_Family":
            "Representation MMD",

        "Outcome":
            row[
                "Outcome"
            ],

        "Class":
            row[
                "Class"
            ],

        "Friedman_Q":
            row[
                "Friedman_Q"
            ],

        "Exact_P":
            row[
                "Exact_P"
            ],

        "Kendalls_W":
            row[
                "Kendalls_W"
            ],

        "Holm_P":
            row[
                "Holm_P"
            ],

        "Supported_After_Holm":
            row[
                "Supported_After_Holm"
            ]
    })


final_statistical_summary_df = (
    pd.DataFrame(
        final_statistical_rows
    )
)


display(
    final_statistical_summary_df
)

In [ ]:
# ============================================
# Section 12.2
# Supported Corrected Phase 5 Findings
# ============================================

supported_findings_df = (

    final_statistical_summary_df[

        final_statistical_summary_df[
            "Supported_After_Holm"
        ]
        ==
        True
    ]

    .copy()
)


display(
    supported_findings_df
)


print(
    "\nNumber of supported outcomes:",
    len(
        supported_findings_df
    )
)

## Section 12.3: Corrected Phase 5 Inferential Conclusions

Two predefined outcomes remained statistically supported after multiplicity correction.

### 1. Malignant predictive-entropy disparity

Training composition significantly altered the Fair–Dark predictive-entropy difference for malignant lesions:

$$
Q=10.0
$$

$$
p_{\mathrm{exact}}=0.000772
$$

$$
W=1.00
$$

$$
p_{\mathrm{Holm}}=0.003086.
$$

This indicates a highly consistent training-composition effect on relative predictive uncertainty for malignant lesions across all five matched seeds.

### 2. Malignant class-conditional Brier disparity

Training composition also significantly altered the Fair–Dark Brier-score difference for malignant lesions:

$$
Q=8.4
$$

$$
p_{\mathrm{exact}}=0.008488
$$

$$
W=0.84
$$

$$
p_{\mathrm{Holm}}=0.016975.
$$

Thus, training composition affected the relative probabilistic prediction error between Fair and Dark malignant lesions.

No multiplicity-adjusted evidence was found for a composition effect on:

- benign malignant-probability disparity,
- malignant malignant-probability disparity,
- benign predictive-entropy disparity,
- benign class-conditional Brier disparity,
- benign avgpool MMD²,
- malignant avgpool MMD².

Therefore, the corrected analysis does not support a simple interpretation in which increasing Dark representation uniformly reduces all forms of domain discrepancy.

## Section 12.4: Malignant Error-Persistence Interpretation

The Dark malignant false-negative analysis demonstrated a persistent image-level difficulty structure.

Mean false-negative recurrence across five seeds decreased descriptively from:

$$
1.79
\rightarrow
1.53
\rightarrow
1.51
$$

for M1, M2, and M3 respectively.

The number of Dark malignant images misclassified by all five seeds decreased from:

$$
13
\rightarrow
8
\rightarrow
6.
$$

However, false-negative persistence remained strongly correlated across compositions, with Spearman correlations ranging approximately from:

$$
0.68
\text{ to }
0.79.
$$

Persistent hard-case Jaccard overlap ranged approximately from:

$$
0.38
\text{ to }
0.46.
$$

Thus, increasing Dark representation modified the frequency and severity of malignant classification failures but did not eliminate a stable subset of repeatedly difficult Dark malignant lesions.

These image-level results are descriptive and are not treated as independent confirmatory replications.

## Section 12.5: Learned Representation Interpretation

Corrected avgpool representation analysis used class-conditional unbiased Gaussian-kernel MMD² with common class-specific bandwidths.

Neither benign nor malignant Fair–Dark representation discrepancy remained statistically supported after Holm correction.

For benign lesions:

$$
p_{\mathrm{Holm}}=0.186728
$$

with:

$$
W=0.52.
$$

For malignant lesions:

$$
p_{\mathrm{Holm}}=0.367284
$$

with:

$$
W=0.28.
$$

Descriptively, benign MMD² increased across M1, M2, and M3, whereas malignant MMD² decreased.

These trends should not be interpreted as statistically established effects.

The absence of a supported avgpool MMD effect also means that the supported malignant entropy and Brier effects cannot be attributed to a demonstrated global change in final-layer Fair–Dark representation discrepancy.

In [ ]:
# ============================================
# Section 13.1
# Publication Figure
# Malignant Predictive Entropy Difference
# ============================================

entropy_plot_df = (

    within_model_domain_difference_df[

        within_model_domain_difference_df[
            "Class"
        ]
        ==
        "Malignant"
    ]

    .copy()
)


plt.figure(
    figsize=(7, 5.5)
)


for seed in SEEDS:

    seed_df = (

        entropy_plot_df[

            entropy_plot_df[
                "Seed"
            ]
            ==
            seed
        ]

        .set_index(
            "Experiment"
        )

        .loc[
            EXPERIMENT_ORDER
        ]

        .reset_index()
    )


    plt.plot(

        range(
            len(
                EXPERIMENT_ORDER
            )
        ),

        seed_df[
            "Predictive_Entropy_Difference"
        ],

        marker="o",

        alpha=0.55
    )


mean_entropy = (

    entropy_plot_df

    .groupby(
        "Experiment"
    )[
        "Predictive_Entropy_Difference"
    ]

    .mean()

    .reindex(
        EXPERIMENT_ORDER
    )
)


plt.plot(

    range(
        len(
            EXPERIMENT_ORDER
        )
    ),

    mean_entropy.values,

    marker="o",

    linewidth=3,

    label="Mean"
)


plt.axhline(
    0,
    linewidth=1
)


plt.xticks(

    range(
        len(
            EXPERIMENT_ORDER
        )
    ),

    [
        "M1\n75F/25D",
        "M2\n50F/50D",
        "M3\n25F/75D"
    ]
)


plt.ylabel(
    "Fair − Dark Predictive Entropy"
)

plt.xlabel(
    "Training Composition"
)

plt.title(
    "Malignant Predictive-Entropy Domain Difference"
)

plt.legend()

plt.tight_layout()


entropy_figure_file = (

    PHASE5_FIGURE_PATH
    / "Phase5_Corrected_Malignant_Entropy_Domain_Difference.png"
)


plt.savefig(

    entropy_figure_file,

    dpi=300,

    bbox_inches="tight"
)


plt.show()


print(
    entropy_figure_file
)

In [ ]:
# ============================================
# Section 13.2
# Publication Figure
# Malignant Brier Domain Difference
# ============================================

brier_plot_df = (

    class_brier_difference_df[

        class_brier_difference_df[
            "Class"
        ]
        ==
        "Malignant"
    ]

    .copy()
)


plt.figure(
    figsize=(7, 5.5)
)


for seed in SEEDS:

    seed_df = (

        brier_plot_df[

            brier_plot_df[
                "Seed"
            ]
            ==
            seed
        ]

        .set_index(
            "Experiment"
        )

        .loc[
            EXPERIMENT_ORDER
        ]

        .reset_index()
    )


    plt.plot(

        range(
            len(
                EXPERIMENT_ORDER
            )
        ),

        seed_df[
            "Brier_Difference"
        ],

        marker="o",

        alpha=0.55
    )


mean_brier = (

    brier_plot_df

    .groupby(
        "Experiment"
    )[
        "Brier_Difference"
    ]

    .mean()

    .reindex(
        EXPERIMENT_ORDER
    )
)


plt.plot(

    range(
        len(
            EXPERIMENT_ORDER
        )
    ),

    mean_brier.values,

    marker="o",

    linewidth=3,

    label="Mean"
)


plt.axhline(
    0,
    linewidth=1
)


plt.xticks(

    range(
        len(
            EXPERIMENT_ORDER
        )
    ),

    [
        "M1\n75F/25D",
        "M2\n50F/50D",
        "M3\n25F/75D"
    ]
)


plt.ylabel(
    "Fair − Dark Brier Score"
)

plt.xlabel(
    "Training Composition"
)

plt.title(
    "Malignant Class-Conditional Brier Difference"
)

plt.legend()

plt.tight_layout()


brier_figure_file = (

    PHASE5_FIGURE_PATH
    / "Phase5_Corrected_Malignant_Brier_Domain_Difference.png"
)


plt.savefig(

    brier_figure_file,

    dpi=300,

    bbox_inches="tight"
)


plt.show()


print(
    brier_figure_file
)

In [ ]:
# ============================================
# Section 14.1
# Final Corrected Phase 5 Artifact Audit
# ============================================

required_phase5_files = [

    PHASE5_PREDICTION_PATH
    / "Phase5_Corrected_All_Model_Predictions.csv",

    PHASE5_TABLE_PATH
    / "Phase5_Corrected_Model_Level_Calibration.csv",

    PHASE5_TABLE_PATH
    / "Phase5_Corrected_Class_Conditional_Brier.csv",

    PHASE5_TABLE_PATH
    / "Phase5_Corrected_Class_Conditional_Brier_Exact_Friedman.csv",

    PHASE5_TABLE_PATH
    / "Phase5_Corrected_Malignant_Entropy_Posthoc.csv",

    PHASE5_TABLE_PATH
    / "Phase5_Corrected_Malignant_Entropy_Bootstrap.csv",

    PHASE5_TABLE_PATH
    / "Phase5_Corrected_Malignant_Brier_Posthoc.csv",

    PHASE5_TABLE_PATH
    / "Phase5_Corrected_Malignant_Brier_Bootstrap.csv",

    PHASE5_TABLE_PATH
    / "Phase5_Corrected_Malignant_Error_Persistence.csv",

    PHASE5_TABLE_PATH
    / "Phase5_Corrected_Malignant_Error_Changes.csv",

    PHASE5_TABLE_PATH
    / "Phase5_Corrected_Malignant_Error_Persistence_Correlation.csv",

    PHASE5_TABLE_PATH
    / "Phase5_Corrected_Malignant_Hard_Case_Overlap.csv",

    PHASE5_TABLE_PATH
    / "Phase5_Corrected_Representation_Manifest.csv",

    PHASE5_TABLE_PATH
    / "Phase5_Corrected_Common_MMD_Bandwidths.csv",

    PHASE5_TABLE_PATH
    / "Phase5_Corrected_Class_Conditional_Representation_Shift.csv",

    PHASE5_TABLE_PATH
    / "Phase5_Corrected_Common_Bandwidth_MMD_Exact_Friedman.csv",

    PHASE5_TABLE_PATH
    / "Phase5_Corrected_PCA_Manifest.csv",

    PHASE5_TABLE_PATH
    / "Phase5_Corrected_PCA_Variance_Summary.csv"
]


artifact_audit_rows = []


for file_path in (
    required_phase5_files
):

    artifact_audit_rows.append({

        "File":
            str(
                file_path
            ),

        "Exists":
            file_path.exists()
    })


final_artifact_audit_df = (
    pd.DataFrame(
        artifact_audit_rows
    )
)


display(
    final_artifact_audit_df
)


print(
    "\nFiles present:",
    int(
        final_artifact_audit_df[
            "Exists"
        ]
        .sum()
    ),

    "/",

    len(
        final_artifact_audit_df
    )
)

In [ ]:
# ============================================
# Section 14.2
# Save Final Statistical Summary
# ============================================

final_statistical_summary_df.to_csv(

    PHASE5_TABLE_PATH
    / "Phase5_Corrected_Final_Statistical_Summary.csv",

    index=False
)


supported_findings_df.to_csv(

    PHASE5_TABLE_PATH
    / "Phase5_Corrected_Supported_Findings.csv",

    index=False
)


final_artifact_audit_df.to_csv(

    PHASE5_AUDIT_PATH
    / "Phase5_Corrected_Final_Artifact_Audit.csv",

    index=False
)


print(
    "✓ Final statistical and artifact audit files saved."
)

In [ ]:
# ============================================
# Section 14.3
# Create Corrected Phase 5 Completion Manifest
# ============================================

phase5_completion_manifest = {

    "Phase":
        "Corrected Phase 5",

    "Title":
        (
            "Mechanistic Analysis of Skin-Tone "
            "Domain Shift and Learned Representations"
        ),

    "Status":
        "COMPLETE",

    "Upstream_Phase":
        "Corrected Phase 4",

    "Experiments":
        EXPERIMENT_ORDER,

    "Seeds":
        SEEDS,

    "Number_of_Models":
        15,

    "Prediction_Records":
        int(
            len(
                all_predictions_df
            )
        ),

    "Representation_Artifacts":
        30,

    "Representation_Dimension":
        2048,

    "Representation_Audit_Passed":
        bool(
            (
                representation_audit_df[
                    "Exists"
                ]

                &

                representation_audit_df[
                    "Shape_Correct"
                ]

                &

                representation_audit_df[
                    "Metadata_Correct"
                ]

                &

                representation_audit_df[
                    "Finite"
                ]
            )
            .all()
        ),

    "Major_Artifact_Audit_Passed":
        bool(
            final_artifact_audit_df[
                "Exists"
            ]
            .all()
        ),

    "Supported_Inferential_Outcomes":
        int(
            len(
                supported_findings_df
            )
        ),

    "Supported_Findings": [

        {
            "Outcome":
                "Malignant predictive entropy Fair-Dark difference",

            "Exact_P":
                0.000772,

            "Holm_P":
                0.003086,

            "Kendalls_W":
                1.00
        },

        {
            "Outcome":
                "Malignant class-conditional Brier Fair-Dark difference",

            "Exact_P":
                0.008488,

            "Holm_P":
                0.016975,

            "Kendalls_W":
                0.84
        }
    ],

    "Representation_MMD_Supported":
        False,

    "PCA_Interpretation":
        "Descriptive only",

    "Branch_Status":
        "Authoritative corrected Phase 5 branch",

    "Frozen":
        True
}


completion_manifest_file = (

    PHASE5_MANIFEST_PATH

    / "Phase5_Corrected_Completion_Manifest.json"
)


with open(
    completion_manifest_file,
    "w"
) as f:

    json.dump(

        phase5_completion_manifest,

        f,

        indent=4
    )


print(
    "✓ Completion manifest saved:"
)

print(
    completion_manifest_file
)

In [ ]:
# ============================================
# Section 14.4
# Final Phase 5 Completion Check
# ============================================

print(
    "=" * 70
)

print(
    "CORRECTED PHASE 5 COMPLETION STATUS"
)

print(
    "=" * 70
)


print(
    "Models analyzed:",
    15
)

print(
    "Prediction records:",
    len(
        all_predictions_df
    )
)

print(
    "Representation artifacts:",
    "30 / 30"
)

print(
    "Supported inferential outcomes:",
    len(
        supported_findings_df
    )
)

print(
    "Major artifact audit:",
    final_artifact_audit_df[
        "Exists"
    ].all()
)

print(
    "Representation audit:",
    phase5_completion_manifest[
        "Representation_Audit_Passed"
    ]
)

print(
    "Status:",
    phase5_completion_manifest[
        "Status"
    ]
)

print(
    "Frozen:",
    phase5_completion_manifest[
        "Frozen"
    ]
)